# 🔬 BUSI Complete Pipeline
**Breast Ultrasound Image (BUSI) Dataset** — End-to-end pipeline covering:
1. EDA & Preprocessing
2. Data Augmentation & DataLoaders
3. Baseline Models: CNN (VGG-16)
4. Multi-Model Training: ViT, U-Net, CNN, ResNet
5. XAI: Grad-CAM for ViT & U-Net

**Dataset**: [BUSI with GT (Kaggle)](https://www.kaggle.com/datasets/aryashah2k/breast-ultrasound-images-dataset) — 780 images, 3 classes (Normal / Benign / Malignant)

---
## 📁 Part 1: EDA, Preprocessing & Augmentation

### 1.1 Setup & Installation

In [ ]:
!pip install -q timm albumentations scikit-learn matplotlib seaborn opencv-python-headless

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import random
import cv2
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import warnings
import pandas as pd
warnings.filterwarnings('ignore')

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Using device: {device}')

# Set seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
print('✅ Seed set for reproducibility')

### 1.2 Dataset Upload & Path Setup
Upload `Dataset_BUSI_with_GT.zip` from Kaggle when prompted.

In [ ]:
import os

# ---- Option A: Upload via Colab file picker ----
from google.colab import files
print('📂 Please upload Dataset_BUSI_with_GT.zip')
uploaded = files.upload()

# Unzip
!unzip -q Dataset_BUSI_with_GT.zip -d /content/dataset

# ---- Auto-detect DATA_ROOT ----
def find_data_root(base='/content/dataset'):
    """
    Automatically find the directory that contains benign/malignant/normal subfolders.
    Handles different zip structures.
    """
    required = {'benign', 'malignant', 'normal'}
    for root, dirs, _ in os.walk(base):
        if required.issubset(set(d.lower() for d in dirs)):
            return root
    return None

DATA_ROOT = find_data_root()

if DATA_ROOT is None:
    raise FileNotFoundError(
        '❌ Could not find benign/malignant/normal folders.\n'
        'Make sure the zip contains Dataset_BUSI_with_GT with three subfolders.'
    )

print(f'✅ DATA_ROOT found: {DATA_ROOT}')

# Verify
total = 0
for cls in ['benign', 'malignant', 'normal']:
    cls_dir = os.path.join(DATA_ROOT, cls)
    imgs = [f for f in os.listdir(cls_dir) if f.endswith('.png') and '_mask' not in f]
    print(f'  {cls:12s}: {len(imgs):4d} images')
    total += len(imgs)
print(f'  {"TOTAL":12s}: {total:4d} images')

### 1.3 Dataset Statistics

In [ ]:
# -------------------------------------------------------
# Collect full dataset metadata
# -------------------------------------------------------
dataset_info = []

for cls in ['benign', 'malignant', 'normal']:
    cls_dir = Path(DATA_ROOT) / cls
    img_files = sorted([f for f in cls_dir.iterdir()
                        if f.suffix == '.png' and '_mask' not in f.name])
    for img_path in img_files:
        img = Image.open(img_path)
        w, h = img.size
        arr = np.array(img.convert('L'))
        masks = list(cls_dir.glob(f'{img_path.stem}_mask*.png'))
        n_masks = len(masks) if cls != 'normal' else 0
        dataset_info.append({
            'class': cls,
            'filename': img_path.name,
            'width': w,
            'height': h,
            'aspect_ratio': round(w / h, 3),
            'mean_intensity': round(arr.mean(), 2),
            'std_intensity': round(arr.std(), 2),
            'n_masks': n_masks,
        })

print('='*55)
print('       BUSI DATASET DESCRIPTION SUMMARY')
print('='*55)
print(f'Total images          : {len(dataset_info)}')

class_counts = Counter(d['class'] for d in dataset_info)
for cls, cnt in class_counts.items():
    pct = cnt / len(dataset_info) * 100
    print(f'  {cls:12s}      : {cnt:4d}  ({pct:.1f}%)')

widths  = [d['width']  for d in dataset_info]
heights = [d['height'] for d in dataset_info]
means   = [d['mean_intensity'] for d in dataset_info]

print(f'\nImage Width  — min:{min(widths)}, max:{max(widths)}, mean:{np.mean(widths):.0f}')
print(f'Image Height — min:{min(heights)}, max:{max(heights)}, mean:{np.mean(heights):.0f}')
print(f'Mean Pixel Intensity — min:{min(means):.1f}, max:{max(means):.1f}, avg:{np.mean(means):.1f}')
print('='*55)

In [ ]:
# -------------------------------------------------------
# Statistical Plots
# -------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('BUSI Dataset — Statistical Overview', fontsize=16, fontweight='bold', y=1.01)

# 1. Class distribution bar
cls_labels = list(class_counts.keys())
cls_vals   = list(class_counts.values())
colors = ['#2196F3', '#FF5722', '#4CAF50']
bars = axes[0,0].bar(cls_labels, cls_vals, color=colors, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, cls_vals):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(val),
                   ha='center', va='bottom', fontweight='bold')
axes[0,0].set_title('Class Distribution', fontweight='bold')
axes[0,0].set_ylabel('Number of Images')
axes[0,0].set_xlabel('Class')

# 2. Class distribution pie
axes[0,1].pie(cls_vals, labels=cls_labels, autopct='%1.1f%%',
              colors=colors, startangle=90,
              wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0,1].set_title('Class Proportion', fontweight='bold')

# 3. Image resolution scatter
for cls, color in zip(['benign', 'malignant', 'normal'], colors):
    subset = [d for d in dataset_info if d['class'] == cls]
    axes[0,2].scatter([d['width'] for d in subset],
                      [d['height'] for d in subset],
                      label=cls, alpha=0.6, color=color, s=30)
axes[0,2].set_title('Image Resolutions', fontweight='bold')
axes[0,2].set_xlabel('Width (px)')
axes[0,2].set_ylabel('Height (px)')
axes[0,2].legend()

# 4. Pixel intensity distribution per class
for cls, color in zip(['benign', 'malignant', 'normal'], colors):
    subset_means = [d['mean_intensity'] for d in dataset_info if d['class'] == cls]
    axes[1,0].hist(subset_means, bins=20, alpha=0.6, label=cls, color=color, edgecolor='white')
axes[1,0].set_title('Mean Pixel Intensity per Class', fontweight='bold')
axes[1,0].set_xlabel('Mean Intensity')
axes[1,0].set_ylabel('Frequency')
axes[1,0].legend()

# 5. Image height distribution
axes[1,1].hist(heights, bins=25, color='#9C27B0', edgecolor='white', alpha=0.8)
axes[1,1].axvline(np.mean(heights), color='red', linestyle='--', label=f'Mean: {np.mean(heights):.0f}px')
axes[1,1].set_title('Image Height Distribution', fontweight='bold')
axes[1,1].set_xlabel('Height (px)')
axes[1,1].set_ylabel('Count')
axes[1,1].legend()

# 6. Std intensity
for cls, color in zip(['benign', 'malignant', 'normal'], colors):
    stds = [d['std_intensity'] for d in dataset_info if d['class'] == cls]
    axes[1,2].hist(stds, bins=20, alpha=0.6, label=cls, color=color, edgecolor='white')
axes[1,2].set_title('Pixel Intensity Std Dev per Class', fontweight='bold')
axes[1,2].set_xlabel('Std Deviation')
axes[1,2].set_ylabel('Count')
axes[1,2].legend()

plt.tight_layout()
plt.savefig('dataset_statistics.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Statistics plot saved as dataset_statistics.png')

In [ ]:
def load_first_mask(mask_paths):
    """Load and combine all mask files for one image."""
    if not mask_paths:
        return None
    combined = None
    for mp in mask_paths:
        m = np.array(Image.open(mp).convert('L'))
        combined = m if combined is None else np.maximum(combined, m)
    return combined


# 7. Lesion Size Distribution

lesion_data = []
for item in dataset_info:
    if item['class'] != 'normal' and item['n_masks'] > 0:
        cls_dir = Path(DATA_ROOT) / item['class']
        img_path = cls_dir / item['filename']
        mask_paths = list(cls_dir.glob(f'{img_path.stem}_mask*.png'))

        # Load the original image dimensions to create a zero-mask of the right size
        img_orig = Image.open(img_path)
        mask_original_shape = img_orig.size[::-1] # (height, width)

        # Use the _load_mask static method from BUSIDataset
        # For simplicity, let's just use the load_first_mask helper previously defined
        loaded_mask = load_first_mask([str(m) for m in mask_paths])

        if loaded_mask is not None:
            # Calculate lesion area (sum of non-zero pixels)
            lesion_area = np.sum(loaded_mask > 127) # Count pixels greater than 127 (mask white pixels)
            lesion_data.append({
                'class': item['class'],
                'lesion_area': lesion_area
            })

if not lesion_data:
    print('⚠️ No lesions with masks found to calculate area distribution.')
else:
    lesion_df = pd.DataFrame(lesion_data)

    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle('BUSI Dataset — Lesion Area Distribution (Pixels)', fontsize=14, fontweight='bold')

    # Plotting histogram for each class
    sns.histplot(data=lesion_df, x='lesion_area', hue='class', kde=True, ax=ax,
                 palette={'benign': '#2196F3', 'malignant': '#FF5722'})

    ax.set_xlabel('Lesion Area (pixels)')
    ax.set_ylabel('Frequency')
    ax.legend(title='Class')
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig('lesion_area_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('📊 Lesion area distribution plot saved as lesion_area_distribution.png')

In [ ]:
# 1. Class Distribution Bar Plot

fig, ax = plt.subplots(figsize=(8, 6))
fig.suptitle('BUSI Dataset — Class Distribution', fontsize=14, fontweight='bold')

cls_labels = list(class_counts.keys())
cls_vals   = list(class_counts.values())
colors = ['#2196F3', '#FF5722', '#4CAF50']
bars = ax.bar(cls_labels, cls_vals, color=colors, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, cls_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(val),
            ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Number of Images')
ax.set_xlabel('Class')

plt.tight_layout()
plt.savefig('class_distribution_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Class distribution bar plot saved as class_distribution_bar.png')

In [ ]:
# 2. Class Distribution Pie Plot

fig, ax = plt.subplots(figsize=(8, 6))
fig.suptitle('BUSI Dataset — Class Proportion', fontsize=14, fontweight='bold')

cls_labels = list(class_counts.keys())
cls_vals   = list(class_counts.values())
colors = ['#2196F3', '#FF5722', '#4CAF50']
ax.pie(cls_vals, labels=cls_labels, autopct='%1.1f%%',
       colors=colors, startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})

plt.tight_layout()
plt.savefig('class_proportion_pie.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Class proportion pie plot saved as class_proportion_pie.png')

In [ ]:
# 3. Image Resolution Scatter Plot

fig, ax = plt.subplots(figsize=(10, 8))
fig.suptitle('BUSI Dataset — Image Resolutions', fontsize=14, fontweight='bold')

colors = ['#2196F3', '#FF5722', '#4CAF50']
for cls, color in zip(['benign', 'malignant', 'normal'], colors):
    subset = [d for d in dataset_info if d['class'] == cls]
    ax.scatter([d['width'] for d in subset],
               [d['height'] for d in subset],
               label=cls, alpha=0.6, color=color, s=30)
ax.set_xlabel('Width (px)')
ax.set_ylabel('Height (px)')
ax.legend()

plt.tight_layout()
plt.savefig('image_resolution_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Image resolution scatter plot saved as image_resolution_scatter.png')

In [ ]:
# 4. Pixel Intensity Distribution per Class Plot

fig, ax = plt.subplots(figsize=(10, 8))
fig.suptitle('BUSI Dataset — Mean Pixel Intensity per Class', fontsize=14, fontweight='bold')

colors = ['#2196F3', '#FF5722', '#4CAF50']
for cls, color in zip(['benign', 'malignant', 'normal'], colors):
    subset_means = [d['mean_intensity'] for d in dataset_info if d['class'] == cls]
    ax.hist(subset_means, bins=20, alpha=0.6, label=cls, color=color, edgecolor='white')
ax.set_xlabel('Mean Intensity')
ax.set_ylabel('Frequency')
ax.legend()

plt.tight_layout()
plt.savefig('pixel_intensity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Mean pixel intensity distribution plot saved as pixel_intensity_distribution.png')

In [ ]:
# 5. Image Height Distribution Plot

fig, ax = plt.subplots(figsize=(8, 6))
fig.suptitle('BUSI Dataset — Image Height Distribution', fontsize=14, fontweight='bold')

ax.hist(heights, bins=25, color='#9C27B0', edgecolor='white', alpha=0.8)
ax.axvline(np.mean(heights), color='red', linestyle='--', label=f'Mean: {np.mean(heights):.0f}px')
ax.set_xlabel('Height (px)')
ax.set_ylabel('Count')
ax.legend()

plt.tight_layout()
plt.savefig('image_height_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Image height distribution plot saved as image_height_distribution.png')

In [ ]:
# 6. Pixel Intensity Standard Deviation per Class Plot

fig, ax = plt.subplots(figsize=(10, 8))
fig.suptitle('BUSI Dataset — Pixel Intensity Std Dev per Class', fontsize=14, fontweight='bold')

colors = ['#2196F3', '#FF5722', '#4CAF50']
for cls, color in zip(['benign', 'malignant', 'normal'], colors):
    stds = [d['std_intensity'] for d in dataset_info if d['class'] == cls]
    ax.hist(stds, bins=20, alpha=0.6, label=cls, color=color, edgecolor='white')
ax.set_xlabel('Std Deviation')
ax.set_ylabel('Count')
ax.legend()

plt.tight_layout()
plt.savefig('pixel_intensity_std_dev.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Pixel intensity standard deviation plot saved as pixel_intensity_std_dev.png')

### 1.4 Data Visualization
Shows 3 samples per class (image | mask | overlay) with bounding boxes, plus per-class pixel intensity histograms.

In [ ]:
def get_bbox_from_mask(mask_arr):
    """Return (x_min, y_min, x_max, y_max) in pixel coords, or None."""
    binary = mask_arr > 127
    if not binary.any():
        return None
    rows = np.any(binary, axis=1)
    cols = np.any(binary, axis=0)
    y_min, y_max = np.where(rows)[0][[0, -1]]
    x_min, x_max = np.where(cols)[0][[0, -1]]
    return (x_min, y_min, x_max, y_max)


def load_first_mask(mask_paths):
    """Load and combine all mask files for one image."""
    if not mask_paths:
        return None
    combined = None
    for mp in mask_paths:
        m = np.array(Image.open(mp).convert('L'))
        combined = m if combined is None else np.maximum(combined, m)
    return combined

In [ ]:
# -------------------------------------------------------
# Visualize 3 samples per class: Image | Mask | Overlay
# -------------------------------------------------------
CLASSES = ['benign', 'malignant', 'normal']
N_PER_CLASS = 3

fig, axes = plt.subplots(len(CLASSES) * N_PER_CLASS, 3,
                          figsize=(14, 5 * len(CLASSES)))
fig.suptitle('BUSI Dataset Samples — Image | Mask | Overlay + BBox',
             fontsize=15, fontweight='bold', y=1.01)

col_titles = ['Original Image', 'Segmentation Mask', 'Overlay + BBox']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold', pad=8)

row = 0
for cls in CLASSES:
    cls_dir = Path(DATA_ROOT) / cls
    img_files = sorted([f for f in cls_dir.iterdir()
                        if f.suffix == '.png' and '_mask' not in f.name])
    sampled = random.sample(img_files, min(N_PER_CLASS, len(img_files)))

    for img_path in sampled:
        img = np.array(Image.open(img_path).convert('RGB'))
        mask_paths = list(cls_dir.glob(f'{img_path.stem}_mask*.png'))
        mask = load_first_mask([str(m) for m in mask_paths]) if cls != 'normal' else None

        # Col 0: original
        axes[row, 0].imshow(img)
        axes[row, 0].set_ylabel(f'{cls}\n{img_path.name}', fontsize=8, rotation=0,
                                 labelpad=80, va='center')
        axes[row, 0].axis('off')

        # Col 1: mask
        if mask is not None:
            axes[row, 1].imshow(mask, cmap='hot')
        else:
            axes[row, 1].imshow(np.zeros(img.shape[:2]), cmap='gray')
            axes[row, 1].text(img.shape[1]//2, img.shape[0]//2, 'No Lesion',
                               ha='center', va='center', color='white', fontsize=10)
        axes[row, 1].axis('off')

        # Col 2: overlay
        overlay = img.copy()
        if mask is not None:
            mask_resized = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
            colored_mask = np.zeros_like(img)
            colored_mask[:, :, 0] = (mask_resized > 127) * 255  # red channel
            overlay = cv2.addWeighted(img, 0.7, colored_mask, 0.3, 0)
            bbox = get_bbox_from_mask(mask_resized)
            if bbox:
                x1, y1, x2, y2 = bbox
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                          linewidth=2, edgecolor='lime', facecolor='none')
                axes[row, 2].add_patch(rect)
        axes[row, 2].imshow(overlay)
        axes[row, 2].axis('off')

        row += 1

plt.tight_layout()
plt.savefig('sample_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('🖼️ Sample visualization saved as sample_visualization.png')

In [ ]:
# -------------------------------------------------------
# Pixel intensity histograms: one sample per class
# -------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Pixel Intensity Histograms per Class', fontsize=14, fontweight='bold')

for ax, cls, color in zip(axes, CLASSES, colors):
    cls_dir = Path(DATA_ROOT) / cls
    img_files = [f for f in cls_dir.iterdir() if f.suffix == '.png' and '_mask' not in f.name]
    sample_path = random.choice(img_files)
    arr = np.array(Image.open(sample_path).convert('L'))
    ax.hist(arr.flatten(), bins=50, color=color, alpha=0.8, edgecolor='none')
    ax.set_title(f'{cls.capitalize()}', fontweight='bold')
    ax.set_xlabel('Pixel Intensity (0–255)')
    ax.set_ylabel('Frequency')
    ax.axvline(arr.mean(), color='black', linestyle='--', label=f'μ={arr.mean():.0f}')
    ax.legend()

plt.tight_layout()
plt.savefig('intensity_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.5 Preprocessing Pipeline
Steps: `resize(224×224) → CLAHE (clip=2.0) → ImageNet normalize → ToTensor`

CLAHE boosts contrast in low-signal ultrasound images before feeding to pretrained models.

In [ ]:
# -------------------------------------------------------
# Preprocessing pipeline
# Steps: resize → CLAHE → normalize → tensor
# -------------------------------------------------------
IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def preprocess_pipeline(split='train'):
    """
    Returns an Albumentations transform pipeline.
    - CLAHE applied to boost contrast in ultrasound images
    - ImageNet normalization for compatibility with pretrained models
    """
    base = [
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),  # Contrast enhancement
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ]
    return A.Compose(base, additional_targets={'mask': 'mask'})


# Visualize preprocessing effect
cls_dir = Path(DATA_ROOT) / 'benign'
sample_path = random.choice([f for f in cls_dir.iterdir()
                              if f.suffix == '.png' and '_mask' not in f.name])
orig = np.array(Image.open(sample_path).convert('RGB'))

# Apply CLAHE manually for visual comparison
gray   = cv2.cvtColor(orig, cv2.COLOR_RGB2GRAY)
clahe  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
enhanced = clahe.apply(gray)
resized = cv2.resize(orig, (IMAGE_SIZE, IMAGE_SIZE))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Preprocessing Steps for One Sample Image', fontsize=14, fontweight='bold')

axes[0].imshow(orig);          axes[0].set_title(f'Original\n{orig.shape[1]}×{orig.shape[0]}');      axes[0].axis('off')
axes[1].imshow(resized);       axes[1].set_title(f'Resized\n{IMAGE_SIZE}×{IMAGE_SIZE}');              axes[1].axis('off')
axes[2].imshow(enhanced, cmap='gray'); axes[2].set_title('CLAHE Enhanced\n(Contrast++)');           axes[2].axis('off')

# Show histogram equalized effect
hist_eq = cv2.equalizeHist(gray)
axes[3].imshow(hist_eq, cmap='gray'); axes[3].set_title('Histogram Equalized\n(Comparison)');        axes[3].axis('off')

plt.tight_layout()
plt.savefig('preprocessing_steps.png', dpi=150, bbox_inches='tight')
plt.show()
print('🔧 Preprocessing visualization saved.')

In [ ]:
# -------------------------------------------------------
# Compute dataset-level mean & std for normalization
# -------------------------------------------------------
print('Computing dataset mean and std (this may take a minute)...')

pixel_sum  = np.zeros(3, dtype=np.float64)
pixel_sq   = np.zeros(3, dtype=np.float64)
pixel_count = 0

for cls in CLASSES:
    cls_dir = Path(DATA_ROOT) / cls
    for img_path in cls_dir.iterdir():
        if img_path.suffix != '.png' or '_mask' in img_path.name:
            continue
        img = np.array(Image.open(img_path).convert('RGB')).astype(np.float64) / 255.0
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        pixel_sum  += img.sum(axis=(0, 1))
        pixel_sq   += (img ** 2).sum(axis=(0, 1))
        pixel_count += IMAGE_SIZE * IMAGE_SIZE

dataset_mean = pixel_sum  / pixel_count
dataset_std  = np.sqrt(pixel_sq / pixel_count - dataset_mean ** 2)

print(f'\n📐 BUSI Dataset Normalization Stats:')
print(f'  Mean : R={dataset_mean[0]:.4f}, G={dataset_mean[1]:.4f}, B={dataset_mean[2]:.4f}')
print(f'  Std  : R={dataset_std[0]:.4f},  G={dataset_std[1]:.4f},  B={dataset_std[2]:.4f}')
print(f'\n  ImageNet Mean (reference): {IMAGENET_MEAN}')
print(f'  ImageNet Std  (reference): {IMAGENET_STD}')
print('\n  ℹ️  Using ImageNet stats is fine for transfer learning.')

### 1.6 Data Augmentation
Medical-image-safe augmentations — no color distortions that would break clinical meaning.
- **Spatial**: HorizontalFlip, VerticalFlip, RandomRotate90, ShiftScaleRotate, RandomResizedCrop
- **Intensity**: Brightness/Contrast, GaussianBlur, GaussNoise

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# -------------------------------------------------------
# Augmentation pipeline definitions
# -------------------------------------------------------

def get_train_augmentation(image_size=224):
    """
    Strong augmentation for training set.
    Medical image safe — no color distortions that break clinical meaning.
    """
    return A.Compose([
        # Spatial
        A.Resize(image_size, image_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
        # Zoom/crop
        A.RandomResizedCrop(size=(image_size, image_size),
                             scale=(0.85, 1.0), ratio=(0.9, 1.1), p=0.4),
        # Intensity — safe for ultrasound
        A.CLAHE(clip_limit=2.0, p=0.4),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        A.GaussNoise(var_limit=(5.0, 20.0), p=0.3),
        # Normalize + tensor
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ], additional_targets={'mask': 'mask'})


def get_val_augmentation(image_size=224):
    """Minimal transform for validation/test — no randomness."""
    return A.Compose([
        A.Resize(image_size, image_size),
        A.CLAHE(clip_limit=2.0, p=1.0),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ], additional_targets={'mask': 'mask'})


print('✅ Augmentation pipelines defined:')
print('   • get_train_augmentation() — 10 spatial + intensity transforms')
print('   • get_val_augmentation()   — resize + CLAHE + normalize only')


In [ ]:
# -------------------------------------------------------
# Augmentation comparison: 8 augmented versions of one image
# -------------------------------------------------------
cls_dir    = Path(DATA_ROOT) / 'benign'
sample_img_path = random.choice([f for f in cls_dir.iterdir()
                                  if f.suffix == '.png' and '_mask' not in f.name])
mask_paths  = list(cls_dir.glob(f'{sample_img_path.stem}_mask*.png'))

orig_img  = np.array(Image.open(sample_img_path).convert('RGB'))
orig_mask = load_first_mask([str(m) for m in mask_paths])
if orig_mask is None:
    orig_mask = np.zeros(orig_img.shape[:2], dtype=np.uint8)

# Use augmentation WITHOUT normalization+tensor for display
display_aug = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
    A.GaussianBlur(blur_limit=(3, 5), p=0.4),
    A.GaussNoise(var_limit=(10.0, 30.0), p=0.4),
    A.CLAHE(clip_limit=2.5, p=0.4),
], additional_targets={'mask': 'mask'})

N_AUG = 8
fig, axes = plt.subplots(2, N_AUG // 2 + 1, figsize=(22, 8))
fig.suptitle('Data Augmentation Showcase (Benign Sample)', fontsize=14, fontweight='bold')

# Flatten axes
ax_flat = axes.flatten()

# Row 0: augmented images
orig_resized = cv2.resize(orig_img, (224, 224))
ax_flat[0].imshow(orig_resized)
ax_flat[0].set_title('Original', fontweight='bold', color='red')
ax_flat[0].axis('off')

for i in range(1, N_AUG + 1):
    result = display_aug(image=orig_img, mask=orig_mask)
    aug_img  = result['image']
    aug_mask = result['mask']

    ax_flat[i].imshow(aug_img)
    # Overlay mask
    if aug_mask.max() > 0:
        contours, _ = cv2.findContours((aug_mask > 127).astype(np.uint8),
                                        cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        overlay_img = aug_img.copy()
        cv2.drawContours(overlay_img, contours, -1, (0, 255, 0), 2)
        ax_flat[i].imshow(overlay_img)
    ax_flat[i].set_title(f'Aug #{i}', fontsize=9)
    ax_flat[i].axis('off')

# Hide any remaining axes
for j in range(N_AUG + 1, len(ax_flat)):
    ax_flat[j].axis('off')

plt.tight_layout()
plt.savefig('augmentation_showcase.png', dpi=150, bbox_inches='tight')
plt.show()
print('🔄 Augmentation showcase saved as augmentation_showcase.png')

In [ ]:
# -------------------------------------------------------
# Per-transform effect: show each augmentation separately
# -------------------------------------------------------
single_transforms = {
    'Original': None,
    'Horizontal Flip': A.HorizontalFlip(p=1.0),
    'Vertical Flip': A.VerticalFlip(p=1.0),
    'Rotate 90': A.RandomRotate90(p=1.0),
    'Affine': A.Affine(translate_percent={'x': (-0.05, 0.05), 'y': (-0.05, 0.05)},
                       scale=(0.9, 1.1), rotate=(-15, 15), p=1.0),
    'CLAHE': A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    'Brightness/Contrast': A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=1.0),
    'Gaussian Blur': A.GaussianBlur(blur_limit=(3, 5), p=1.0),
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Individual Image Augmentations', fontsize=14, fontweight='bold')

for ax, (name, transform) in zip(axes.flat, single_transforms.items()):
    if transform is None:
        img = cv2.resize(orig_img, (IMAGE_SIZE, IMAGE_SIZE))
    else:
        img = transform(image=cv2.resize(orig_img, (IMAGE_SIZE, IMAGE_SIZE)))['image']
    ax.imshow(img)
    ax.set_title(name, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('individual_augmentations.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved individual_augmentations.png')


### 1.7 BUSIDataset Class & DataLoaders
Custom PyTorch `Dataset` supporting **classification | segmentation | detection** simultaneously.
- Class mapping: `{normal: 0, benign: 1, malignant: 2}` — malignant upweighted ×2
- Handles multi-mask files and returns both normalized + pixel bboxes
- Train/val/test splits: 70/15/15

In [ ]:
class BUSIDataset(Dataset):
    """
    BUSI Dataset — Breast Ultrasound Images
    Supports: classification | segmentation | detection

    Fixes applied vs original:
    - Auto-detect DATA_ROOT (no empty-samples crash)
    - Guard for empty samples list before train_test_split
    - Combined multi-mask files properly
    - Bounding box returned in pixel coords as well as normalized
    """

    CLASS_MAPPING = {'normal': 0, 'benign': 1, 'malignant': 2}
    CLASS_WEIGHTS  = {'normal': 1.0, 'benign': 1.0, 'malignant': 2.0}  # upweight minority

    def __init__(self, data_root, split='train', image_size=(224, 224),
                 transform=None, include_normal=True):
        assert split in ('train', 'val', 'test'), f'Invalid split: {split}'
        self.data_root     = Path(data_root)
        self.split         = split
        self.image_size    = image_size
        self.include_normal = include_normal

        all_samples = self._scan_dataset()

        # ── BUGFIX: guard against empty dataset before split ──
        if len(all_samples) == 0:
            raise RuntimeError(
                f'No images found in {data_root}.\n'
                'Make sure the path contains benign/malignant/normal subfolders with .png files.'
            )

        self.samples  = self._split_data(all_samples)
        self.transform = transform if transform is not None else self._default_transform()

        print(f'[BUSIDataset] {split:5s} split: {len(self.samples):4d} samples '
              f'| {self._class_dist()}')

    # ── internal helpers ──────────────────────────────────

    def _scan_dataset(self):
        samples = []
        classes = ['benign', 'malignant'] + (['normal'] if self.include_normal else [])

        for cls in classes:
            cls_dir = self.data_root / cls
            if not cls_dir.exists():
                print(f'⚠️  Folder not found, skipping: {cls_dir}')
                continue

            img_files = sorted([f for f in cls_dir.iterdir()
                                 if f.suffix == '.png' and '_mask' not in f.name])
            for img_path in img_files:
                mask_paths = [] if cls == 'normal' else \
                    sorted(cls_dir.glob(f'{img_path.stem}_mask*.png'))
                samples.append({
                    'image_path': str(img_path),
                    'mask_paths': [str(m) for m in mask_paths],
                    'class_name': cls,
                    'class_id':   self.CLASS_MAPPING[cls],
                    'has_lesion': int(cls != 'normal'),
                    'weight':     self.CLASS_WEIGHTS[cls],
                })
        return samples

    def _split_data(self, samples):
        labels = [s['class_id'] for s in samples]
        # 70 / 15 / 15 stratified split
        train_s, temp_s, _, temp_l = train_test_split(
            samples, labels, test_size=0.30, stratify=labels, random_state=42
        )
        val_s, test_s = train_test_split(
            temp_s, test_size=0.50, stratify=temp_l, random_state=42
        )
        return {'train': train_s, 'val': val_s, 'test': test_s}[self.split]

    def _default_transform(self):
        return get_train_augmentation(self.image_size[0]) if self.split == 'train' \
               else get_val_augmentation(self.image_size[0])

    def _class_dist(self):
        cnt = Counter(s['class_name'] for s in self.samples)
        return '  '.join(f'{k}:{v}' for k, v in sorted(cnt.items()))

    # ── mask & bbox helpers ───────────────────────────────

    @staticmethod
    def _load_mask(mask_paths, shape_hw):
        if not mask_paths:
            return np.zeros(shape_hw, dtype=np.uint8)
        combined = np.zeros(shape_hw, dtype=np.uint8)
        for mp in mask_paths:
            m = np.array(Image.open(mp).convert('L'))
            if m.shape != shape_hw:
                m = np.array(Image.fromarray(m).resize(
                    (shape_hw[1], shape_hw[0]), Image.NEAREST))
            combined = np.maximum(combined, m)
        return (combined > 127).astype(np.uint8) * 255

    @staticmethod
    def _bbox_normalized(mask):
        """Return (x1,y1,x2,y2) normalized to [0,1]. (0,0,0,0) if no lesion."""
        if mask.max() == 0:
            return (0., 0., 0., 0.)
        rows = np.any(mask > 0, axis=1)
        cols = np.any(mask > 0, axis=0)
        y1, y2 = np.where(rows)[0][[0, -1]]
        x1, x2 = np.where(cols)[0][[0, -1]]
        h, w = mask.shape
        return (x1/w, y1/h, x2/w, y2/h)

    # ── Dataset interface ─────────────────────────────────

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s     = self.samples[idx]
        image = np.array(Image.open(s['image_path']).convert('RGB'))
        mask  = self._load_mask(s['mask_paths'], image.shape[:2])
        bbox  = self._bbox_normalized(mask)

        aug   = self.transform(image=image, mask=mask)
        image = aug['image']          # (C, H, W) tensor
        mask  = aug['mask']           # (H, W) tensor or ndarray

        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask)
        mask = (mask.float() / 255.0).unsqueeze(0)   # (1, H, W)

        return {
            'image':      image,
            'mask':       mask,
            'bbox':       torch.tensor(bbox,             dtype=torch.float32),
            'label':      torch.tensor(s['class_id'],    dtype=torch.long),
            'has_lesion': torch.tensor(s['has_lesion'],  dtype=torch.float32),
            'class_name': s['class_name'],
        }

print('✅ BUSIDataset class defined.')

In [ ]:
# -------------------------------------------------------
# Create datasets and dataloaders
# -------------------------------------------------------
BATCH_SIZE = 16
NUM_WORKERS = 2

train_ds = BUSIDataset(DATA_ROOT, split='train')
val_ds   = BUSIDataset(DATA_ROOT, split='val')
test_ds  = BUSIDataset(DATA_ROOT, split='test')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'\n📦 DataLoaders ready:')
print(f'   Train  — {len(train_ds):4d} samples, {len(train_loader):3d} batches')
print(f'   Val    — {len(val_ds):4d} samples, {len(val_loader):3d} batches')
print(f'   Test   — {len(test_ds):4d} samples, {len(test_loader):3d} batches')

# Verify one batch
batch = next(iter(train_loader))
print(f'\n🔍 Batch shapes:')
print(f'   images:  {batch["image"].shape}')
print(f'   masks:   {batch["mask"].shape}')
print(f'   bboxes:  {batch["bbox"].shape}')
print(f'   labels:  {batch["label"].shape}  values: {batch["label"].unique().tolist()}')

### 1.8 Augmented Batch Visualization

In [ ]:
# -------------------------------------------------------
# Visualize one training batch
# -------------------------------------------------------
CLASS_NAMES = {0: 'Normal', 1: 'Benign', 2: 'Malignant'}
batch = next(iter(train_loader))

n_show = min(8, BATCH_SIZE)
fig, axes = plt.subplots(2, n_show, figsize=(22, 5))
fig.suptitle('Training Batch — Augmented Images (top) & Masks (bottom)',
             fontsize=13, fontweight='bold')

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

for i in range(n_show):
    img  = batch['image'][i] * std + mean          # denormalize
    img  = img.permute(1, 2, 0).numpy().clip(0, 1)
    mask = batch['mask'][i].squeeze().numpy()
    lbl  = batch['label'][i].item()

    axes[0, i].imshow(img)
    axes[0, i].set_title(CLASS_NAMES[lbl], fontsize=8,
                          color=['blue','orange','red'][lbl], fontweight='bold')
    axes[0, i].axis('off')

    axes[1, i].imshow(mask, cmap='hot')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('training_batch.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training batch visualization saved as training_batch.png')

In [ ]:
print('='*60)
print('  PIPELINE SUMMARY')
print('='*60)
print(f'  Dataset   : BUSI (Breast Ultrasound Images)')
print(f'  Classes   : Normal | Benign | Malignant')
print(f'  Train/Val/Test split: 70 / 15 / 15  (stratified)')
print(f'  Image size: {IMAGE_SIZE}×{IMAGE_SIZE}')
print(f'  Preprocessing: Resize → CLAHE → ImageNet Norm')
print(f'  Augmentation (train): HFlip + VFlip + Rotate + ShiftScale')
print(f'                        + RandomCrop + Brightness/Contrast')
print(f'                        + GaussianBlur + GaussNoise + CLAHE')
print(f'  Batch size: {BATCH_SIZE}')
print('='*60)

---
## 📁 Part 2: Baseline Models — CNN (VGG-16)

### 2.1 MultiTaskCNN — VGG-16 Backbone (Pretrained)
Classification head: `GlobalAvgPool → LayerNorm → Linear(512,256) → GELU → Dropout → Linear(256,3)`

In [ ]:
"""
Classification-only CNN (VGG-16) for breast lesion analysis
Baseline comparison model using VGG-16 backbone with a classification head
"""

import torch
import torch.nn as nn
from typing import Dict
import torchvision.models as models


class CNNBackbone(nn.Module):
    """VGG-16 backbone for feature extraction"""

    def __init__(self, pretrained: bool = True):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT if pretrained else None)
        features = list(vgg.features.children())

        self.block1 = nn.Sequential(*features[:5])
        self.block2 = nn.Sequential(*features[5:10])
        self.block3 = nn.Sequential(*features[10:17])
        self.block4 = nn.Sequential(*features[17:24])
        self.block5 = nn.Sequential(*features[24:31])
        self.global_pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        global_features = self.global_pool(x).flatten(1)
        return {"global_features": global_features}


class CNNClassificationHead(nn.Module):
    """Classification head for CNN global features"""

    def __init__(self, in_features: int = 512, num_classes: int = 3, dropout: float = 0.1):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Linear(in_features, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(x)


class MultiTaskLoss(nn.Module):
    """Classification loss"""

    def __init__(self, cls_weight=1.0):
        super().__init__()
        cls_weights = torch.tensor([1.0, 1.0, 2.0])
        self.cls_criterion = nn.CrossEntropyLoss(weight=cls_weights)
        self.cls_weight = cls_weight

    def forward(self, predictions, targets):
        cls_loss = self.cls_criterion(predictions['cls_logits'], targets['label'])
        total = self.cls_weight * cls_loss
        return total, {'cls': cls_loss}


class MultiTaskCNN(nn.Module):
    """
    CNN classifier for Breast Ultrasound Analysis
    """

    def __init__(self, pretrained: bool = True, num_classes: int = 3, dropout: float = 0.1, **kwargs):
        super().__init__()
        self.backbone = CNNBackbone(pretrained=pretrained)
        self.classification_head = CNNClassificationHead(512, num_classes, dropout)

    def forward(self, x: torch.Tensor, return_attention: bool = False):
        features = self.backbone(x)
        cls_logits = self.classification_head(features["global_features"])
        return {"cls_logits": cls_logits}

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True

    def get_num_parameters(self):
        backbone_params = sum(p.numel() for p in self.backbone.parameters())
        cls_params = sum(p.numel() for p in self.classification_head.parameters())
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {
            "backbone": backbone_params,
            "classification_head": cls_params,
            "total": total,
            "trainable": trainable,
        }


if __name__ == "__main__":
    model = MultiTaskCNN(pretrained=False)
    x = torch.randn(2, 3, 224, 224)
    output = model(x)
    print(f"Classification logits: {output['cls_logits'].shape}")

    params = model.get_num_parameters()
    for k, v in params.items():
        print(f"  {k}: {v:,}")


### 2.2 MultiTaskRNN — CNN+BiLSTM Hybrid (From Scratch)
Treats spatial feature maps as sequences: `CNN → flatten rows → BiLSTM → classification head`

### 2.3 Metrics Tracker & Attention Explainer

In [ ]:
class MetricsTracker:
    """Tracks classification predictions and targets across batches."""
    def __init__(self):
        self.cls_preds = []
        self.cls_targets = []

    def update(self, predictions, targets):
        self.cls_preds.extend(predictions['cls_logits'].detach().argmax(dim=1).cpu().numpy())
        self.cls_targets.extend(targets['label'].detach().cpu().numpy())

    def compute(self):
        from sklearn.metrics import accuracy_score, precision_recall_fscore_support
        acc = accuracy_score(self.cls_targets, self.cls_preds)
        p, r, f1, _ = precision_recall_fscore_support(
            self.cls_targets, self.cls_preds, average='weighted', zero_division=0)
        return {
            'cls_accuracy': acc,
            'cls_precision': p,
            'cls_recall': r,
            'cls_f1': f1,
        }

    def get_classification_report(self):
        from sklearn.metrics import classification_report
        target_names = ['Normal', 'Benign', 'Malignant']
        return classification_report(
            self.cls_targets, self.cls_preds, target_names=target_names, zero_division=0)

    def get_confusion_matrix(self):
        from sklearn.metrics import confusion_matrix
        return confusion_matrix(self.cls_targets, self.cls_preds)


class AttentionExplainer:
    """Generates attention-based explanations using the model's forward pass."""
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def visualize_explanation(self, image_tensor, save_path=None, show=False):
        image_tensor = image_tensor.to(self.device)
        self.model.eval()
        with torch.no_grad():
            output = self.model(image_tensor)
            probabilities = torch.softmax(output['cls_logits'][0], dim=0).cpu()

        class_name = ["Normal", "Benign", "Malignant"][probabilities.argmax().item()]
        print(f"[Explainer] Predicted: {class_name} "
              f"(conf: {probabilities.max():.2%})")
        if show:
            print("  (Visualization not rendered in placeholder mode)")
        if save_path:
            print(f"  (Explanation image would be saved to: {save_path})")
        return {
            'class_name': class_name,
            'probabilities': probabilities
        }

    def visualize_head_attention(self, image_tensor, layer_idx=-1, save_path=None, show=False):
        print(f"[Explainer] Visualizing head attention for layer {layer_idx}")
        if show:
            print("  (Visualization not rendered in placeholder mode)")
        if save_path:
            print(f"  (Attention heads image would be saved to: {save_path})")


class DummyConfig:
    """Placeholder for a configuration object"""
    def __init__(self):
        self.data = DummyDataConfig()
        self.model = DummyModelConfig()


class DummyDataConfig:
    def __init__(self):
        self.data_root = '/content/dataset/Dataset_BUSI_with_GT'


class DummyModelConfig:
    def __init__(self):
        self.vit_model = 'vit_base_patch16_224'
        self.num_classes = 3


def get_config():
    """Placeholder for get_config function"""
    return DummyConfig()


### 2.4 Training Loop (CNN Baseline)

In [ ]:
import argparse
import os
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

SAVE_DIR = "./checkpoints"


def evaluate_model(model, test_loader, device):
    """Run classification evaluation on test set"""
    model.eval()
    metrics = MetricsTracker()

    with torch.no_grad():
        for batch in test_loader:
            images = batch["image"].to(device)
            targets = {
                "label": batch["label"].to(device),
                "has_lesion": batch["has_lesion"].to(device),
            }
            predictions = model(images, return_attention=False)
            metrics.update(predictions, targets)

    results = metrics.compute()

    print("\n" + "=" * 60)
    print("EVALUATION RESULTS")
    print("=" * 60)

    print("\n--- Classification ---")
    print(f"  Accuracy:    {results['cls_accuracy']:.4f}")
    print(f"  Precision:   {results['cls_precision']:.4f}")
    print(f"  Recall:      {results['cls_recall']:.4f}")
    print(f"  F1 Score:    {results['cls_f1']:.4f}")

    print("\n--- Detailed Classification Report ---")
    print(metrics.get_classification_report())

    print("--- Confusion Matrix ---")
    print(metrics.get_confusion_matrix())

    return results


def generate_explanations(model, test_loader, device, output_dir, num_samples=10):
    """Generate explainability visualizations for sample images"""
    explainer = AttentionExplainer(model, device)
    output_dir = Path(output_dir) / "explanations"
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nGenerating explanations for {num_samples} samples...")

    count = 0
    for batch in test_loader:
        for i in range(batch["image"].shape[0]):
            if count >= num_samples:
                break

            image = batch["image"][i:i+1].to(device)
            label = batch["label"][i].item()
            class_name = ["Normal", "Benign", "Malignant"][label]

            save_path = output_dir / f"sample_{count}_gt_{class_name}.png"
            result = explainer.visualize_explanation(image, save_path=str(save_path), show=False)

            head_save_path = output_dir / f"sample_{count}_heads.png"
            explainer.visualize_head_attention(image, layer_idx=-1, save_path=str(head_save_path), show=False)

            print(f"  Sample {count}: GT={class_name}, Pred={result['class_name']} "
                  f"(conf: {result['probabilities'].max():.2%})")

            count += 1

        if count >= num_samples:
            break

    print(f"\nExplanations saved to: {output_dir}")


def predict_single_image(model, image_path, device):
    """Run prediction on a single image with explanation"""
    transform = get_val_augmentation(image_size=224)

    image = np.array(Image.open(image_path).convert("RGB"))
    transformed = transform(image=image)
    image_tensor = transformed["image"].unsqueeze(0).to(device)

    model.eval()
    explainer = AttentionExplainer(model, device)
    save_path = str(Path(image_path).parent / f"{Path(image_path).stem}_explanation.png")

    result = explainer.visualize_explanation(image_tensor, save_path=save_path, show=True)

    print(f"\nPrediction: {result['class_name']}")
    print(f"Confidence: {result['probabilities'].max():.2%}")
    print(f"Probabilities: Normal={result['probabilities'][0]:.2%}, "
          f"Benign={result['probabilities'][1]:.2%}, "
          f"Malignant={result['probabilities'][2]:.2%}")
    print(f"Explanation saved to: {save_path}")

    return result


def run_evaluation_and_explanation():
    global test_loader, device

    checkpoint_path = os.path.join(SAVE_DIR, 'best_cnn.pth')
    if not os.path.exists(checkpoint_path):
        print("WARNING: No trained checkpoint found at", checkpoint_path)
        print("WARNING: Using random (untrained) weights - evaluation metrics are MEANINGLESS.")
        os.makedirs(SAVE_DIR, exist_ok=True)
        model = MultiTaskCNN(pretrained=False, num_classes=3)
        checkpoint_path = os.path.join(SAVE_DIR, 'dummy_checkpoint.pth')
        torch.save({"model_state_dict": model.state_dict()}, checkpoint_path)
    else:
        model = MultiTaskCNN(pretrained=False, num_classes=3)

    print(f"Loading model from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    print("Model loaded successfully")

    evaluate_model(model, test_loader, device)

    output_dir = "./results"
    os.makedirs(output_dir, exist_ok=True)
    generate_explanations(model, test_loader, device, output_dir, num_samples=3)


if False:
    run_evaluation_and_explanation()


In [ ]:
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. Define hyperparameters for training
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

print(f'\nStarting training loop for {NUM_EPOCHS} epochs...')

# 2. Instantiate the MultiTaskCNN model
model = MultiTaskCNN(pretrained=True, num_classes=3).to(device)

# 3. Instantiate the MultiTaskLoss function
loss_fn = MultiTaskLoss(cls_weight=1.0).to(device)

# 4. Instantiate an Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# 5a. Initialize variables to track the best validation accuracy and save path
best_val_acc = -1.0
checkpoint_path = os.path.join(SAVE_DIR, 'best_cnn.pth')

# Create directory if it doesn't exist
os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(1, NUM_EPOCHS + 1):
    # 5c. Set the model to training mode
    model.train()
    total_train_loss = 0.0

    # 5d. Iterate through the train_loader
    for batch_idx, batch in enumerate(train_loader):
        images = batch['image'].to(device)
        targets = {
            'label': batch['label'].to(device),
            'has_lesion': batch['has_lesion'].to(device)
        }

        # 5d.ii. Perform a forward pass
        predictions = model(images)

        # 5d.iii. Calculate the total loss
        loss, _ = loss_fn(predictions, targets)

        # 5d.iv. Perform backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 5d.v. Accumulate training loss
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # 5e. Set the model to evaluation mode
    model.eval()
    total_val_loss = 0.0
    metrics_tracker = MetricsTracker()

    # 5f. Iterate through the val_loader with torch.no_grad()
    with torch.no_grad():
        for batch_idx, batch in enumerate(val_loader):
            images = batch['image'].to(device)
            targets = {
                'label': batch['label'].to(device),
                'has_lesion': batch['has_lesion'].to(device)
            }

            # 5f.ii. Perform a forward pass
            predictions = model(images)

            # 5f.iii. Calculate the total loss
            loss, _ = loss_fn(predictions, targets)

            # 5f.iv. Accumulate validation loss
            total_val_loss += loss.item()

            # 5f.v. Track classification predictions and targets
            metrics_tracker.update(predictions, targets)

    avg_val_loss = total_val_loss / len(val_loader)
    val_results = metrics_tracker.compute()

    # 5g. Compute and print results
    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | ' +
          f'Train Loss: {avg_train_loss:.4f} | ' +
          f'Val Loss: {avg_val_loss:.4f} | ' +
          f'Val Acc: {val_results["cls_accuracy"]:.4f} | ' +
          f'Val F1: {val_results["cls_f1"]:.4f}')

    # 5h. If current validation accuracy is better, save model
    if val_results['cls_accuracy'] > best_val_acc:
        best_val_acc = val_results['cls_accuracy']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_val_loss,
            'accuracy': best_val_acc,
        }, checkpoint_path)
        print(f'  --> Saved best model checkpoint to {checkpoint_path} (Acc: {best_val_acc:.4f})')

print('\nTraining complete.')


---
## 📁 Part 3: Multi-Model Training — ViT, U-Net, CNN, ResNet + XAI

### 3.1 Setup
Additional installs for Part 3: `scikit-image`, `segmentation-models-pytorch`

In [ ]:
!pip install -q timm albumentations scikit-learn scikit-image segmentation-models-pytorch


In [ ]:
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.models as tv_models

import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, precision_score, recall_score
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
SAVE_DIR   = '/content/checkpoints'
RESULTS_DIR= '/content/results_seg'
BATCH_SIZE = 8
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
PATIENCE   = 10
IMAGE_SIZE = (224, 224)
NUM_CLASSES= 3
SEG_WEIGHT = 1.5   # higher weight for segmentation loss
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS_TO_TRAIN = ['vit', 'unet', 'cnn', 'resnet']   # Added 'resnet' to train all 4 models
CLASS_NAMES= ['Normal', 'Benign', 'Malignant']

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Device: {DEVICE}')
print(f'Models to train: {MODELS_TO_TRAIN}')
print(f'Seg loss weight: {SEG_WEIGHT}')

### 3.2 Dataset & DataLoaders

In [ ]:
# ============================================================
# BUSI Dataset
# ============================================================

class BUSIDataset(Dataset):
    CLASS_MAPPING = {"normal": 0, "benign": 1, "malignant": 2}

    def __init__(self, data_root, split="train", image_size=(224, 224), transform=None, include_normal=True):
        self.data_root = Path(data_root)
        self.split = split
        self.image_size = image_size
        self.include_normal = include_normal
        self.samples = self._load_dataset()
        self.transform = transform if transform is not None else self._get_default_transform(split)

    def _load_dataset(self):
        samples = []
        classes = ["benign", "malignant"]
        if self.include_normal:
            classes.append("normal")

        for class_name in classes:
            class_dir = self.data_root / class_name
            if not class_dir.exists():
                print(f"Warning: {class_dir} does not exist, skipping.")
                continue
            image_files = [f for f in class_dir.iterdir()
                           if f.suffix.lower() == '.png' and '_mask' not in f.name]
            for img_path in image_files:
                mask_pattern = img_path.stem + "_mask"
                masks = list(class_dir.glob(f"{mask_pattern}*.png"))
                if class_name == "normal":
                    masks = []
                samples.append({
                    "image_path": str(img_path),
                    "mask_paths": [str(m) for m in masks],
                    "class_name": class_name,
                    "class_id": self.CLASS_MAPPING[class_name],
                })
        return self._split_dataset(samples)

    def _split_dataset(self, samples):
        labels = [s["class_id"] for s in samples]
        train_s, temp_s, _, temp_l = train_test_split(
            samples, labels, test_size=0.3, stratify=labels, random_state=42)
        val_s, test_s = train_test_split(
            temp_s, test_size=0.5, stratify=temp_l, random_state=42)
        return {"train": train_s, "val": val_s, "test": test_s}[self.split]

    def _get_default_transform(self, split):
        if split == "train":
            return A.Compose([
                A.Resize(self.image_size[0], self.image_size[1]),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Affine(
                    translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)},
                    scale=(0.9, 1.1), rotate=(-15, 15), p=0.5),
                A.ElasticTransform(alpha=120, sigma=120*0.05, p=0.3),
                A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
                A.OneOf([
                    A.GaussNoise(),
                    A.GaussianBlur(blur_limit=3),
                    A.MedianBlur(blur_limit=3),
                ], p=0.3),
                A.OneOf([
                    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2),
                    A.CLAHE(clip_limit=2),
                ], p=0.3),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ])
        else:
            return A.Compose([
                A.Resize(self.image_size[0], self.image_size[1]),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ])

    def _load_image(self, path):
        return np.array(Image.open(path).convert("RGB"))

    def _load_mask(self, paths, image_shape):
        if not paths:
            return np.zeros(image_shape[:2], dtype=np.uint8)
        combined = np.zeros(image_shape[:2], dtype=np.uint8)
        for path in paths:
            mask = np.array(Image.open(path).convert("L"))
            if mask.shape != image_shape[:2]:
                mask = np.array(Image.fromarray(mask).resize(
                    (image_shape[1], image_shape[0]), Image.NEAREST))
            combined = np.maximum(combined, mask)
        return (combined > 127).astype(np.uint8) * 255

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = self._load_image(sample["image_path"])
        mask = self._load_mask(sample["mask_paths"], image.shape)
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]
        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask)
        mask = mask.float() / 255.0
        if mask.dim() == 2:
            mask = mask.unsqueeze(0)
        return {
            "image": image,
            "mask": mask,
            "label": torch.tensor(sample["class_id"], dtype=torch.long),
            "image_path": sample["image_path"]
        }


def get_dataloaders(data_root, batch_size=8, image_size=(224, 224), num_workers=2, include_normal=True):
    train_ds = BUSIDataset(data_root, split="train", image_size=image_size, include_normal=include_normal)
    val_ds   = BUSIDataset(data_root, split="val",   image_size=image_size, include_normal=include_normal)
    test_ds  = BUSIDataset(data_root, split="test",  image_size=image_size, include_normal=include_normal)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader

print("Dataset classes defined.")


In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(DATA_ROOT, batch_size=BATCH_SIZE, num_workers=2)
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}")
batch = next(iter(train_loader))
print(f"Image: {batch['image'].shape}, Mask: {batch['mask'].shape}, Labels: {batch['label']}")


### 3.3 Loss Functions
- **FocalTverskyLoss** (`α=0.7, β=0.3, γ=0.75`): Penalizes false negatives — ideal for small lesions
- **DiceLoss**: Standard overlap-based segmentation loss

In [ ]:


class FocalTverskyLoss(nn.Module):
    """Penalizes false negatives more (alpha>beta) — great for small lesions."""
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.gamma = gamma
        self.smooth= smooth

    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred_flat   = pred.view(-1)
        target_flat = target.view(-1)
        tp = (pred_flat * target_flat).sum()
        fp = (pred_flat * (1 - target_flat)).sum()
        fn = ((1 - pred_flat) * target_flat).sum()
        tversky = (tp + self.smooth) / (tp + self.alpha*fn + self.beta*fp + self.smooth)
        return (1 - tversky) ** self.gamma


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred_flat   = pred.view(-1)
        target_flat = target.view(-1)
        intersection = (pred_flat * target_flat).sum()
        dice = (2.0*intersection + self.smooth) / (pred_flat.sum() + target_flat.sum() + self.smooth)
        return 1.0 - dice


class CombinedSegLoss(nn.Module):
    """BCE + FocalTversky (replaces BCE+Dice for better small-lesion recall)"""
    def __init__(self, bce_weight=0.4, focal_tversky_weight=0.6):
        super().__init__()
        self.bce          = nn.BCEWithLogitsLoss()
        self.focal_tversky= FocalTverskyLoss(alpha=0.7, beta=0.3, gamma=0.75)
        self.bce_weight   = bce_weight
        self.ft_weight    = focal_tversky_weight

    def forward(self, pred, target):
        return self.bce_weight * self.bce(pred, target) + self.ft_weight * self.focal_tversky(pred, target)


class MultiTaskLoss(nn.Module):
    def __init__(self, cls_weight=1.0, seg_weight=1.5):
        super().__init__()
        self.classification_loss = nn.CrossEntropyLoss()
        self.segmentation_loss   = CombinedSegLoss()
        self.cls_weight = cls_weight
        self.seg_weight = seg_weight

    def forward(self, predictions, targets):
        cls_loss = self.classification_loss(predictions['cls_logits'], targets['label'])
        seg_loss = self.segmentation_loss(predictions['seg_mask'], targets['mask'])
        total    = self.cls_weight * cls_loss + self.seg_weight * seg_loss
        return {'total': total, 'classification': cls_loss, 'segmentation': seg_loss}

print('Loss functions defined (FocalTversky + BCE).')


### 3.4 Metrics Tracker (Multi-Task)

In [ ]:
# ============================================================
# Metrics Tracker
# ============================================================

class MetricsTracker:
    def __init__(self, num_classes=3, class_names=None):
        self.num_classes = num_classes
        self.class_names = class_names or ["Normal", "Benign", "Malignant"]
        self.reset()

    def reset(self):
        self.all_cls_preds = []
        self.all_cls_labels = []
        self.seg_dice_scores = []
        self.seg_iou_scores = []

    def update(self, predictions, targets):
        # Classification
        cls_probs = torch.softmax(predictions["cls_logits"].detach().cpu(), dim=-1)
        cls_preds = cls_probs.argmax(dim=-1)
        self.all_cls_preds.extend(cls_preds.numpy().tolist())
        self.all_cls_labels.extend(targets["label"].cpu().numpy().tolist())

        # Segmentation
        seg_pred = (torch.sigmoid(predictions["seg_mask"].detach().cpu()) > 0.5).float()
        seg_target = targets["mask"].cpu()
        for i in range(seg_pred.shape[0]):
            self.seg_dice_scores.append(self._compute_dice(seg_pred[i], seg_target[i]))
            self.seg_iou_scores.append(self._compute_iou(seg_pred[i], seg_target[i]))

    def compute(self):
        preds = np.array(self.all_cls_preds)
        labels = np.array(self.all_cls_labels)
        results = {
            "cls_accuracy": accuracy_score(labels, preds),
            "cls_f1": f1_score(labels, preds, average="weighted", zero_division=0),
            "cls_precision": precision_score(labels, preds, average="weighted", zero_division=0),
            "cls_recall": recall_score(labels, preds, average="weighted", zero_division=0),
        }
        per_class_f1 = f1_score(labels, preds, average=None, zero_division=0)
        for i, name in enumerate(self.class_names):
            if i < len(per_class_f1):
                results[f"cls_f1_{name.lower()}"] = float(per_class_f1[i])
        if self.seg_dice_scores:
            results["seg_dice"] = float(np.mean(self.seg_dice_scores))
            results["seg_iou"] = float(np.mean(self.seg_iou_scores))
        return results

    def get_confusion_matrix(self):
        return confusion_matrix(np.array(self.all_cls_labels), np.array(self.all_cls_preds))

    def get_classification_report(self):
        return classification_report(
            np.array(self.all_cls_labels), np.array(self.all_cls_preds),
            target_names=self.class_names, zero_division=0)

    @staticmethod
    def _compute_dice(pred, target):
        p = pred.view(-1).numpy()
        t = target.view(-1).numpy()
        inter = (p * t).sum()
        total = p.sum() + t.sum()
        if total == 0:
            return 1.0
        return float(2.0 * inter / (total + 1e-6))

    @staticmethod
    def _compute_iou(pred, target):
        p = pred.view(-1).numpy()
        t = target.view(-1).numpy()
        inter = (p * t).sum()
        union = p.sum() + t.sum() - inter
        if union == 0:
            return 1.0
        return float(inter / (union + 1e-6))

print("MetricsTracker defined.")

### 3.5 Model Architectures

| Model | Backbone | Pretrained | Tasks |
|---|---|---|---|
| MultiTaskViT | `vit_base_patch16_224` (timm) | ✅ ImageNet | Classification + Segmentation |
| MultiTaskUNet | ResNet-50 encoder (SMP) | ✅ ImageNet | Segmentation + Classification |
| MultiTaskCNN | VGG-16 | ❌ Scratch | Classification |
| MultiTaskResNet | ResNet-50 | ❌ Scratch | Classification |

In [ ]:
# ============================================================
# Shared Task Heads (ViT)
# ============================================================

class ClassificationHead(nn.Module):
    """LayerNorm -> Linear(embed_dim,256) -> GELU -> Dropout -> Linear(256,num_classes)"""
    def __init__(self, embed_dim=768, num_classes=3, dropout=0.1):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.head(x)


class SegmentationHead(nn.Module):
    """Reshape patches to 14x14, then 4x ConvTranspose2d to 224x224"""
    def __init__(self, embed_dim=768, img_size=224, patch_size=16, out_channels=1):
        super().__init__()
        self.grid_size = img_size // patch_size

        self.project = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 512),
            nn.GELU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 2, stride=2),
            nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256, 128, 2, stride=2),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128, 64, 2, stride=2),
            nn.BatchNorm2d(64), nn.GELU(),
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, out_channels, 1)
        )

    def forward(self, patch_tokens):
        B = patch_tokens.shape[0]
        x = self.project(patch_tokens)
        x = x.transpose(1, 2).reshape(B, 512, self.grid_size, self.grid_size)
        return self.decoder(x)

print("Shared task heads defined.")

In [ ]:
# ============================================================
# Model 1: ViT (pretrained)
# ============================================================

class ViTBackbone(nn.Module):
    def __init__(self, model_name="vit_base_patch16_224", pretrained=True,
                 img_size=224, drop_rate=0.1, attn_drop_rate=0.1):
        super().__init__()
        self.vit = timm.create_model(
            model_name, pretrained=pretrained, img_size=img_size,
            drop_rate=drop_rate, attn_drop_rate=attn_drop_rate, num_classes=0)
        self.embed_dim = self.vit.embed_dim
        self.patch_size = self.vit.patch_embed.patch_size[0]
        self.num_patches = (img_size // self.patch_size) ** 2
        self.grid_size = img_size // self.patch_size

    def _get_attention_weights(self, x):
        attention_weights = []
        x = self.vit.patch_embed(x)
        cls_token = self.vit.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.vit.pos_embed
        x = self.vit.pos_drop(x)
        for block in self.vit.blocks:
            B, N, C = x.shape
            norm_x = block.norm1(x)
            qkv = block.attn.qkv(norm_x)
            qkv = qkv.reshape(B, N, 3, block.attn.num_heads, C // block.attn.num_heads)
            qkv = qkv.permute(2, 0, 3, 1, 4)
            q, k, v = qkv.unbind(0)
            attn = (q @ k.transpose(-2, -1)) * (C // block.attn.num_heads) ** -0.5
            attn = attn.softmax(dim=-1)
            attention_weights.append(attn.detach())
            x = block(x)
        return attention_weights

    def forward(self, x, return_attention=True):
        if return_attention:
            attn_weights = self._get_attention_weights(x.clone())
        else:
            attn_weights = None
        features = self.vit.forward_features(x)
        cls_token = features[:, 0]
        patch_tokens = features[:, 1:]
        out = {"cls_token": cls_token, "patch_tokens": patch_tokens,
               "grid_size": self.grid_size}
        if return_attention:
            out["attention_weights"] = attn_weights
        return out


class MultiTaskViT(nn.Module):
    def __init__(self, model_name="vit_base_patch16_224", pretrained=True,
                 num_classes=3, img_size=224, drop_rate=0.1, attn_drop_rate=0.1, **kwargs):
        super().__init__()
        self.backbone = ViTBackbone(model_name, pretrained, img_size, drop_rate, attn_drop_rate)
        embed_dim = self.backbone.embed_dim
        patch_size = self.backbone.patch_size
        self.classification_head = ClassificationHead(embed_dim, num_classes, drop_rate)
        self.segmentation_head = SegmentationHead(embed_dim, img_size, patch_size, 1)

    def forward(self, x, return_attention=True):
        backbone_out = self.backbone(x, return_attention=return_attention)
        cls_logits = self.classification_head(backbone_out["cls_token"])
        seg_mask = self.segmentation_head(backbone_out["patch_tokens"])
        out = {"cls_logits": cls_logits, "seg_mask": seg_mask}
        if return_attention and "attention_weights" in backbone_out:
            out["attention_weights"] = backbone_out["attention_weights"]
        else:
            out["attention_weights"] = None
        return out

    def compute_attention_rollout(self, attention_weights):
        result = None
        for attn in attention_weights:
            attn_mean = attn.mean(dim=1)
            I = torch.eye(attn_mean.size(-1), device=attn_mean.device)
            attn_res = (attn_mean + I) / 2
            attn_res = attn_res / attn_res.sum(dim=-1, keepdim=True)
            result = attn_res if result is None else torch.bmm(attn_res, result)
        return result

    def get_num_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"total": total, "trainable": trainable}

print("Model 1: ViT defined.")

In [ ]:
# NOTE: This MultiTaskCNN (scratch, Part 3) supersedes the pretrained version from Part 2.
# ============================================================
# Model 2: CNN / VGG-16 (from scratch)
# ============================================================

class CNNBackbone(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        weights = tv_models.VGG16_Weights.DEFAULT if pretrained else None
        vgg = tv_models.vgg16(weights=weights)
        features = list(vgg.features.children())
        self.block1 = nn.Sequential(*features[:5])    # 64ch, 112x112
        self.block2 = nn.Sequential(*features[5:10])  # 128ch, 56x56
        self.block3 = nn.Sequential(*features[10:17]) # 256ch, 28x28
        self.block4 = nn.Sequential(*features[17:24]) # 512ch, 14x14
        self.block5 = nn.Sequential(*features[24:31]) # 512ch, 7x7
        self.global_pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x1 = self.block1(x)
        x2 = self.block2(x1)
        x3 = self.block3(x2)
        x4 = self.block4(x3)
        x5 = self.block5(x4)
        global_features = self.global_pool(x5).flatten(1)
        return {"global_features": global_features, "spatial_features": x4}


class CNNClassificationHead(nn.Module):
    def __init__(self, in_features=512, num_classes=3, dropout=0.1):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Linear(in_features, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.head(x)


class CNNSegmentationHead(nn.Module):
    def __init__(self, in_channels=512):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(in_channels, 256, 2, stride=2),
            nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256, 128, 2, stride=2),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128, 64, 2, stride=2),
            nn.BatchNorm2d(64), nn.GELU(),
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, x):
        return self.decoder(x)


class MultiTaskCNN(nn.Module):
    def __init__(self, pretrained=False, num_classes=3, dropout=0.1, **kwargs):
        super().__init__()
        self.backbone = CNNBackbone(pretrained=pretrained)
        self.classification_head = CNNClassificationHead(512, num_classes, dropout)
        self.segmentation_head = CNNSegmentationHead(512)

    def forward(self, x, return_attention=True):
        features = self.backbone(x)
        cls_logits = self.classification_head(features["global_features"])
        seg_mask = self.segmentation_head(features["spatial_features"])
        out = {"cls_logits": cls_logits, "seg_mask": seg_mask}
        if return_attention:
            out["attention_weights"] = None
        return out

    def get_num_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"total": total, "trainable": trainable}

print("Model 2: CNN/VGG-16 defined.")

In [ ]:

# Model 3: ResNet-50 (from scratch)


class ResNetBackbone(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        weights = tv_models.ResNet50_Weights.DEFAULT if pretrained else None
        resnet = tv_models.resnet50(weights=weights)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1   # 256ch, 56x56
        self.layer2 = resnet.layer2   # 512ch, 28x28
        self.layer3 = resnet.layer3   # 1024ch, 14x14
        self.layer4 = resnet.layer4   # 2048ch, 7x7
        self.global_pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        l1 = self.layer1(x)
        l2 = self.layer2(l1)
        l3 = self.layer3(l2)
        l4 = self.layer4(l3)
        global_features = self.global_pool(l4).flatten(1)
        return {"global_features": global_features,
                "layer1": l1, "layer2": l2, "layer3": l3, "layer4": l4}


class ResNetClassificationHead(nn.Module):
    def __init__(self, in_features=2048, num_classes=3, dropout=0.1):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.head(x)


class ResNetSegmentationHead(nn.Module):
    def __init__(self,):
        super().__init__()
        self.up1 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(1024, 512, 3, padding=1), nn.BatchNorm2d(512), nn.GELU())
        self.up2 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.GELU())
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = nn.Sequential(
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.GELU())
        self.up4 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec4 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.GELU())
        self.output = nn.Conv2d(64, 1, 1)

    def forward(self, layer1, layer2, layer3):
        x = self.up1(layer3)
        x = self.dec1(torch.cat([x, layer2], dim=1))
        x = self.up2(x)
        x = self.dec2(torch.cat([x, layer1], dim=1))
        x = self.up3(x)
        x = self.dec3(x)
        x = self.up4(x)
        x = self.dec4(x)
        return self.output(x)


class MultiTaskResNet(nn.Module):
    def __init__(self, pretrained=False, num_classes=3, dropout=0.1, **kwargs):
        super().__init__()
        self.backbone = ResNetBackbone(pretrained=pretrained)
        self.classification_head = ResNetClassificationHead(2048, num_classes, dropout)
        self.segmentation_head = ResNetSegmentationHead()

    def forward(self, x, return_attention=True):
        features = self.backbone(x)
        cls_logits = self.classification_head(features["global_features"])
        seg_mask = self.segmentation_head(features["layer1"], features["layer2"], features["layer3"])
        out = {"cls_logits": cls_logits, "seg_mask": seg_mask}
        if return_attention:
            out["attention_weights"] = None
        return out

    def get_num_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"total": total, "trainable": trainable}

print("Model 3: ResNet-50 defined.")

In [ ]:
# ============================================================
# Model 2: U-Net with Pretrained ResNet-50 Encoder (SMP)
# ============================================================
import segmentation_models_pytorch as smp

class MultiTaskUNet(nn.Module):
    """
    Pretrained ResNet-50 encoder + U-Net decoder for segmentation.
    Classification from encoder bottleneck features.
    Encoder LR = 0.1x head LR (preserves pretrained features).
    """
    def __init__(self, encoder_name='resnet50', encoder_weights='imagenet',
                 num_classes=3, dropout=0.1, **kwargs):
        super().__init__()
        # Segmentation U-Net
        self.unet = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,   # raw logits
        )
        # Classification head on encoder bottleneck (ResNet-50 = 2048 channels)
        enc_out_ch = self.unet.encoder.out_channels[-1]
        self.gap   = nn.AdaptiveAvgPool2d(1)
        self.classification_head = nn.Sequential(
            nn.Linear(enc_out_ch, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x, return_attention=True):
        # Encoder features (list of skip connections)
        features = self.unet.encoder(x)
        # Segmentation
        decoder_out = self.unet.decoder(features)
        seg_mask    = self.unet.segmentation_head(decoder_out)
        # Classification from bottleneck
        bottleneck  = features[-1]                  # (B, 2048, 7, 7)
        cls_feat    = self.gap(bottleneck).flatten(1)
        cls_logits  = self.classification_head(cls_feat)
        out = {'cls_logits': cls_logits, 'seg_mask': seg_mask}
        if return_attention:
            out['attention_weights'] = None
        return out

    def get_num_parameters(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {'total': total, 'trainable': trainable}

print('Model 2: U-Net (ResNet-50 pretrained encoder) defined.')


### 3.6 Optimizer, Scheduler & Model Factory

In [ ]:
# ============================================================
# Model Factory
# ============================================================

def create_model(model_type):
    if model_type == 'vit':
        return MultiTaskViT(pretrained=True, num_classes=NUM_CLASSES)
    elif model_type == 'unet':
        return MultiTaskUNet(encoder_name='resnet50', encoder_weights='imagenet',
                             num_classes=NUM_CLASSES)
    elif model_type == 'cnn':
        # MultiTaskCNN by default uses pretrained=False for VGG-16, as requested 'from scratch'
        return MultiTaskCNN(num_classes=NUM_CLASSES)
    elif model_type == 'resnet':
        # MultiTaskResNet by default uses pretrained=False for ResNet-50
        return MultiTaskResNet(num_classes=NUM_CLASSES)
    else:
        raise ValueError(f'Unknown model type: {model_type}')

# Quick sanity check
print('Testing model creation...')
for mt in MODELS_TO_TRAIN:
    m = create_model(mt)
    params = m.get_num_parameters()
    print(f'  {mt:8s}: {params["total"]:>12,} params  ({params["trainable"]:>12,} trainable)')
    del m
print('All models created successfully.')

In [ ]:

# Optimizer & Scheduler

def create_optimizer(model, model_type, lr=LEARNING_RATE):
    if model_type == 'vit':
        backbone_params = list(model.backbone.parameters())
        head_params = (list(model.classification_head.parameters()) +
                       list(model.segmentation_head.parameters()))
        return torch.optim.AdamW([
            {'params': backbone_params, 'lr': lr * 0.1},
            {'params': head_params,     'lr': lr}
        ], weight_decay=1e-4)
    elif model_type == 'unet':
        # Encoder (pretrained ResNet-50) gets lower LR to preserve features
        encoder_params = list(model.unet.encoder.parameters())
        decoder_params = (list(model.unet.decoder.parameters()) +
                          list(model.unet.segmentation_head.parameters()) +
                          list(model.classification_head.parameters()))
        return torch.optim.AdamW([
            {'params': encoder_params, 'lr': lr * 0.1},
            {'params': decoder_params, 'lr': lr}
        ], weight_decay=1e-4)
    else:
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)


def create_scheduler(optimizer, num_epochs):
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)


# ── Test-Time Augmentation (TTA) ──────────────────────────────
def predict_with_tta(model, images):
    """Average segmentation over original + H-flip + V-flip. Returns averaged seg_mask."""
    model.eval()
    with torch.no_grad():
        out0 = model(images, return_attention=False)
        out1 = model(torch.flip(images, [3]), return_attention=False)   # H-flip
        out2 = model(torch.flip(images, [2]), return_attention=False)   # V-flip
        seg  = (out0['seg_mask'] +
                torch.flip(out1['seg_mask'], [3]) +
                torch.flip(out2['seg_mask'], [2])) / 3.0
        cls  = out0['cls_logits']   # use original for classification
    return {'cls_logits': cls, 'seg_mask': seg}

print('Optimizer, scheduler, and TTA defined.')


### 3.7 Training Loop

In [ ]:
# ============================================================
# Training Function
# ============================================================

def compute_dice(pred_logits, target, threshold=0.5):
    pred  = (torch.sigmoid(pred_logits) > threshold).float()
    inter = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice  = (2*inter + 1e-6) / (union + 1e-6)
    return dice.mean().item()

def compute_iou(pred_logits, target, threshold=0.5):
    pred  = (torch.sigmoid(pred_logits) > threshold).float()
    inter = (pred * target).sum(dim=(1,2,3))
    union = (pred + target).clamp(0,1).sum(dim=(1,2,3))
    iou   = (inter + 1e-6) / (union + 1e-6)
    return iou.mean().item()


def train_model(model_type, train_loader, val_loader, test_loader,
                num_epochs=NUM_EPOCHS, patience=PATIENCE, lr=LEARNING_RATE):
    print(f'\nInitializing {model_type.upper()}...')
    model     = create_model(model_type).to(DEVICE)
    optimizer = create_optimizer(model, model_type, lr)
    scheduler = create_scheduler(optimizer, num_epochs)
    criterion = MultiTaskLoss(cls_weight=1.0, seg_weight=SEG_WEIGHT)

    use_amp = DEVICE.type == 'cuda'
    scaler  = torch.amp.GradScaler('cuda') if use_amp else None

    param_info = model.get_num_parameters()
    print(f'  Parameters: {param_info["total"]:12,} total, {param_info["trainable"]:12,} trainable')

    best_composite = -1.0
    best_epoch     = 0
    patience_count = 0
    history        = {'train_loss':[], 'val_loss':[], 'val_acc':[], 'val_f1':[], 'val_dice':[]}
    ckpt_path      = os.path.join(SAVE_DIR, f'{model_type}_best.pt')

    for epoch in range(1, num_epochs+1):
        # ── TRAIN ──
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            images = batch['image'].to(DEVICE)
            targets = {'label': batch['label'].to(DEVICE),
                       'mask':  batch['mask'].to(DEVICE)}
            optimizer.zero_grad()
            if use_amp:
                with torch.amp.autocast('cuda'):
                    preds  = model(images)
                    losses = criterion(preds, targets)
                scaler.scale(losses['total']).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                preds  = model(images)
                losses = criterion(preds, targets)
                losses['total'].backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            train_loss += losses['total'].item()
        train_loss /= len(train_loader)

        # ── VALIDATE ──
        model.eval()
        val_loss, all_preds, all_labels, val_dices = 0.0, [], [], []
        with torch.no_grad():
            for batch in val_loader:
                images  = batch['image'].to(DEVICE)
                targets = {'label': batch['label'].to(DEVICE),
                           'mask':  batch['mask'].to(DEVICE)}
                preds   = model(images)
                losses  = criterion(preds, targets)
                val_loss += losses['total'].item()
                all_preds.extend(preds['cls_logits'].argmax(1).cpu().tolist())
                all_labels.extend(targets['label'].cpu().tolist())
                val_dices.append(compute_dice(preds['seg_mask'], targets['mask']))
        val_loss /= len(val_loader)
        val_acc   = sum(p==l for p,l in zip(all_preds, all_labels)) / len(all_labels)
        from sklearn.metrics import f1_score
        val_f1    = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
        val_dice  = float(np.mean(val_dices))
        composite = (val_f1 + val_dice) / 2.0

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['val_dice'].append(val_dice)

        scheduler.step()

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Ep {epoch:03d}/{num_epochs}  '
                  f'loss={train_loss:.4f}  val_loss={val_loss:.4f}  '
                  f'acc={val_acc:.4f}  f1={val_f1:.4f}  dice={val_dice:.4f}  '
                  f'composite={composite:.4f}')

        if composite > best_composite:
            best_composite = composite
            best_epoch     = epoch
            patience_count = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f'  Early stopping at epoch {epoch} (best={best_epoch})')
                break

    # ── TEST (with TTA for segmentation) ──
    print(f'\nLoading best checkpoint (epoch {best_epoch}, composite={best_composite:.4f})...')
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    model.eval()

    all_preds, all_labels, all_dices, all_ious = [], [], [], []
    all_seg_preds, all_seg_targets = [], []

    for batch in test_loader:
        images  = batch['image'].to(DEVICE)
        targets = {'label': batch['label'].to(DEVICE), 'mask': batch['mask'].to(DEVICE)}
        # TTA for better segmentation
        preds   = predict_with_tta(model, images)
        all_preds.extend(preds['cls_logits'].argmax(1).cpu().tolist())
        all_labels.extend(targets['label'].cpu().tolist())
        all_dices.append(compute_dice(preds['seg_mask'], targets['mask']))
        all_ious.append(compute_iou(preds['seg_mask'], targets['mask']))
        all_seg_preds.append(preds['seg_mask'].detach().cpu())
        all_seg_targets.append(targets['mask'].detach().cpu())

    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
    test_acc  = accuracy_score(all_labels, all_preds)
    test_f1   = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    test_precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    test_recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    test_dice = float(np.mean(all_dices))
    test_iou  = float(np.mean(all_ious))
    cm        = confusion_matrix(all_labels, all_preds).tolist()
    report    = classification_report(all_labels, all_preds,
                                       target_names=CLASS_NAMES, zero_division=0)

    test_metrics = {
        'cls_accuracy': test_acc, 'cls_f1': test_f1,
        'cls_precision': test_precision, 'cls_recall': test_recall,
        'seg_dice': test_dice,    'seg_iou': test_iou,
        'confusion_matrix': cm
    }
    print(f'\nTest Results for {model_type.upper()}:')
    print(f'  Accuracy : {test_acc:.4f}')
    print(f'  Precision: {test_precision:.4f}')
    print(f'  Recall   : {test_recall:.4f}')
    print(f'  F1 Score : {test_f1:.4f}')
    print(f'  Dice (TTA): {test_dice:.4f}')
    print(f'  IoU  (TTA): {test_iou:.4f}')
    print('\nClassification Report:')
    print(report)

    return {
        'model_type':    model_type,
        'test_results':  test_metrics,
        'history':       history,
        'confusion_matrix': cm,
        'best_epoch':    best_epoch,
        'best_composite':float(best_composite),
        'num_parameters':param_info
    }


# ── Train all models ──────────────────────────────────────────
all_results  = {}
results_path = os.path.join(RESULTS_DIR, 'all_model_results.json')

for model_type in MODELS_TO_TRAIN:
    print(f"\n{'='*60}")
    print(f'  TRAINING: {model_type.upper()}')
    print(f"{'='*60}")
    try:
        results = train_model(model_type, train_loader, val_loader, test_loader)
        all_results[model_type] = results
        with open(results_path, 'w') as f:
            json.dump(all_results, f, indent=2, default=str)
        print(f'>> {model_type.upper()} results saved.')
    except Exception as e:
        print(f'ERROR training {model_type}: {e}')
        import traceback; traceback.print_exc()

print('\nAll models training complete!')

### 3.8 Evaluation & Results

In [ ]:
# ============================================================
# Results Summary Table
# ============================================================

if not all_results:
    # Load from file if needed
    with open(results_path, 'r') as f:
        all_results = json.load(f)

# Ensure MODELS_TO_TRAIN reflects all models for which results are available
# and that results are loaded correctly if not in memory.
# Using the global MODELS_TO_TRAIN as it's updated in the notebook.
models_to_display = MODELS_TO_TRAIN

print(f"{'Model':<14} {'Acc':>8} {'F1':>8} {'Dice':>8} {'IoU':>8} {'Params':>14}")
print("-" * 62)
for model_type in models_to_display:
    if model_type not in all_results:
        print(f"{model_type:<14} {'N/A':>8} {'N/A':>8} {'N/A':>8} {'N/A':>8} {'N/A':>14}")
        continue
    r = all_results[model_type]
    tr = r['test_results']
    acc  = tr.get('cls_accuracy', 0)
    f1   = tr.get('cls_f1', 0)
    # Precision and Recall are part of the detailed classification report
    # but not directly in the aggregated 'test_results' dict for easy display here.
    # We'll display Dice and IoU for segmentation metrics.
    dice = tr.get('seg_dice', 0)
    iou  = tr.get('seg_iou', 0)
    params = r['num_parameters']['total']
    print(f"{model_type:<14} {acc:>8.4f} {f1:>8.4f} {dice:>8.4f} {iou:>8.4f} {params:>14,}")
print("-" * 62)


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import torch
import numpy as np
import os

def plot_roc_curve(model, test_loader, num_classes, class_names, device, save_path=None):
    model.eval()
    all_labels = []
    all_preds_proba = []

    with torch.no_grad():
        for batch in test_loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device)
            preds = model(images)

            # Use softmax for classification probabilities
            probs = torch.softmax(preds['cls_logits'], dim=-1)
            all_labels.extend(labels.cpu().numpy())
            all_preds_proba.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds_proba = np.array(all_preds_proba)

    # Binarize the labels for multi-class ROC
    binarized_labels = label_binarize(all_labels, classes=range(num_classes))

    # Compute ROC curve and ROC area for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(num_classes):
        fpr[i], tpr[i], _ = roc_curve(binarized_labels[:, i], all_preds_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Compute micro-average ROC curve and ROC area
    fpr["micro"], tpr["micro"], _ = roc_curve(binarized_labels.ravel(), all_preds_proba.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    # Compute macro-average ROC curve and ROC area
    # First aggregate all false positive rates
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(num_classes)]))

    # Then interpolate all ROC curves at this points
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(num_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

    # Finally average it and compute AUC
    mean_tpr /= num_classes

    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    # Plot all ROC curves
    plt.figure(figsize=(10, 8))
    plt.plot(fpr["micro"], tpr["micro"],
             label=f'micro-average ROC curve (area = {roc_auc["micro"]:0.2f})',
             color='deeppink', linestyle=':', linewidth=4)

    plt.plot(fpr["macro"], tpr["macro"],
             label=f'macro-average ROC curve (area = {roc_auc["macro"]:0.2f})',
             color='navy', linestyle=':', linewidth=4)

    # Correct way to get N distinct colors from a colormap
    cmap = matplotlib.colormaps.get_cmap('Dark2')
    colors = [cmap(i) for i in np.linspace(0, 1, num_classes)]

    for i, color in zip(range(num_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                 label=f'ROC curve of class {class_names[i]} (area = {roc_auc[i]:0.2f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Chance level (AUC = 0.5)')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic for {model.__class__.__name__}')
    plt.legend(loc="lower right", fontsize='small')
    plt.grid(alpha=0.3)
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"ROC curve saved to: {save_path}")
    plt.show()


print("Generating ROC curves for all models...")
for model_type in MODELS_TO_TRAIN:
    print(f"  Processing {model_type.upper()}...")
    model = create_model(model_type).to(DEVICE)
    ckpt_path = os.path.join(SAVE_DIR, f'{model_type}_best.pt')

    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
        print(f"    {model_type.upper()} checkpoint loaded.")
    else:
        print(f"    Warning: No checkpoint found for {model_type}, using current model weights.")

    roc_save_path = os.path.join(RESULTS_DIR, f'roc_curve_{model_type}.png')
    plot_roc_curve(model, test_loader, NUM_CLASSES, CLASS_NAMES, DEVICE, roc_save_path)
    del model
    torch.cuda.empty_cache() # Clear GPU memory

print("All ROC curves generated.")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np
import torch
import os

def plot_combined_roc_curve(model_types, create_model_func, test_loader, num_classes, class_names, device, save_path=None):
    plt.figure(figsize=(12, 10))
    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Chance level (AUC = 0.5)')

    # Fix: Get the colormap and then sample colors from it
    cmap = matplotlib.colormaps.get_cmap('hsv')
    colors = [cmap(i) for i in np.linspace(0, 1, len(model_types))]

    for i, model_type in enumerate(model_types):
        print(f"  Processing {model_type.upper()} for combined ROC...")
        model = create_model_func(model_type).to(device)
        ckpt_path = os.path.join(SAVE_DIR, f'{model_type}_best.pt')

        if os.path.exists(ckpt_path):
            model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
            print(f"    {model_type.upper()} checkpoint loaded.")
        else:
            print(f"    Warning: No checkpoint found for {model_type}, using current model weights.")

        model.eval()
        all_labels = []
        all_preds_proba = []

        with torch.no_grad():
            for batch in test_loader:
                images = batch['image'].to(device)
                labels = batch['label'].to(device)
                preds = model(images)
                probs = torch.softmax(preds['cls_logits'], dim=-1)
                all_labels.extend(labels.cpu().numpy())
                all_preds_proba.extend(probs.cpu().numpy())

        all_labels = np.array(all_labels)
        all_preds_proba = np.array(all_preds_proba)

        binarized_labels = label_binarize(all_labels, classes=range(num_classes))

        # Compute macro-average ROC curve and ROC area
        fpr_macro = dict()
        tpr_macro = dict()
        roc_auc_macro = dict()

        # For macro-average, compute for each class first
        fpr = dict()
        tpr = dict()
        for j in range(num_classes):
            fpr[j], tpr[j], _ = roc_curve(binarized_labels[:, j], all_preds_proba[:, j])

        # Then aggregate all false positive rates
        all_fpr = np.unique(np.concatenate([fpr[j] for j in range(num_classes)]))

        # Then interpolate all ROC curves at this points
        mean_tpr = np.zeros_like(all_fpr)
        for j in range(num_classes):
            mean_tpr += np.interp(all_fpr, fpr[j], tpr[j])

        # Finally average it and compute AUC
        mean_tpr /= num_classes

        fpr_macro = all_fpr
        tpr_macro = mean_tpr
        roc_auc_macro = auc(fpr_macro, tpr_macro)

        plt.plot(fpr_macro, tpr_macro, color=colors[i], lw=2,
                 label=f'{model_type.upper()} (AUC = {roc_auc_macro:0.2f})')

        del model
        torch.cuda.empty_cache()

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Combined Receiver Operating Characteristic (ROC) for All Models')
    plt.legend(loc="lower right", fontsize='medium')
    plt.grid(alpha=0.3)
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Combined ROC curve saved to: {save_path}")
    plt.show()

print("Generating combined ROC curve for all models...")
combined_roc_save_path = os.path.join(RESULTS_DIR, 'roc_curve_combined_all_models.png')
plot_combined_roc_curve(MODELS_TO_TRAIN, create_model, test_loader, NUM_CLASSES, CLASS_NAMES, DEVICE, combined_roc_save_path)
print("Combined ROC curve generation complete.")

In [ ]:
import seaborn as sns

if not all_results:
    with open(results_path, 'r') as f:
        all_results = json.load(f)

models_to_plot = [m for m in MODELS_TO_TRAIN if m in all_results]
n_models = len(models_to_plot)

# Determine grid size (e.g., 2x2 for 4 models, 2x3 for 5-6 models)
rows = (n_models + 1) // 2 if n_models > 2 else 1
cols = 2 if n_models > 1 else 1
if n_models == 1: # Handle single model case
    rows, cols = 1, 1

fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 5))
fig.suptitle('Classification Confusion Matrices for All Models', fontsize=14, fontweight='bold')

# Flatten axes array for easy iteration if it's a grid
if n_models > 1:
    axes = axes.flatten()
else:
    axes = [axes]

for idx, model_type in enumerate(models_to_plot):
    ax = axes[idx]
    r = all_results[model_type]
    cm = np.array(r['test_results']['confusion_matrix'])

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"{model_type.upper()}") # Simplified title
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

# Hide any unused subplots if n_models is odd and rows*cols > n_models
for i in range(n_models, rows * cols):
    fig.delaxes(axes[i])

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle overlap
plt.savefig(os.path.join(RESULTS_DIR, 'classification_confusion_matrices_all_models.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Classification confusion matrices saved.')


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def compute_segmentation_confusion_matrix(pred_masks, true_masks, threshold=0.5):
    # Flatten the masks to 1D arrays
    pred_flat = (pred_masks > threshold).long().view(-1)
    true_flat = true_masks.long().view(-1)

    # Calculate confusion matrix components
    tp = ((pred_flat == 1) & (true_flat == 1)).sum().item()
    tn = ((pred_flat == 0) & (true_flat == 0)).sum().item()
    fp = ((pred_flat == 1) & (true_flat == 0)).sum().item()
    fn = ((pred_flat == 0) & (true_flat == 1)).sum().item()

    return np.array([[tn, fp], [fn, tp]])


all_segmentation_cms = {}

print("Generating segmentation confusion matrices...")
for model_type in MODELS_TO_TRAIN:
    print(f"  Processing {model_type.upper()}...")
    model = create_model(model_type).to(DEVICE)
    ckpt_path = os.path.join(SAVE_DIR, f'{model_type}_best.pt')

    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    else:
        print(f"    Warning: No checkpoint found for {model_type}, using current model weights.")

    model.eval()

    total_pred_masks = []
    total_true_masks = []

    with torch.no_grad():
        for batch in test_loader:
            images = batch['image'].to(DEVICE)
            true_masks = batch['mask'].to(DEVICE)

            # Use TTA for segmentation predictions
            preds = predict_with_tta(model, images)
            pred_masks = torch.sigmoid(preds['seg_mask'])

            total_pred_masks.append(pred_masks.cpu())
            total_true_masks.append(true_masks.cpu())

    # Concatenate all masks and compute confusion matrix
    all_preds_cat = torch.cat(total_pred_masks, dim=0)
    all_true_masks_cat = torch.cat(total_true_masks, dim=0)

    seg_cm = compute_segmentation_confusion_matrix(all_preds_cat, all_true_masks_cat)
    all_segmentation_cms[model_type] = seg_cm

print("Segmentation confusion matrices generated.")


Visualize segmentation confusion matrices for each model.

In [ ]:
n_models = len(MODELS_TO_TRAIN)
rows = (n_models + 1) // 2 if n_models > 2 else 1
cols = 2 if n_models > 1 else 1
if n_models == 1:
    rows, cols = 1, 1

fig, axes = plt.subplots(rows, cols, figsize=(cols * 7, rows * 6))
fig.suptitle('Segmentation Confusion Matrices (Pixel-wise)', fontsize=16, fontweight='bold', y=1.02)

if n_models > 1:
    axes = axes.flatten()
else:
    axes = [axes]

for idx, model_type in enumerate(MODELS_TO_TRAIN):
    if model_type not in all_segmentation_cms:
        print(f"Skipping {model_type} as segmentation CM was not generated.")
        continue

    ax = axes[idx]
    cm = all_segmentation_cms[model_type]

    # Segmentation Confusion Matrix labels
    cm_labels = [['True Negative', 'False Positive'], ['False Negative', 'True Positive']]

    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Background', 'Lesion'], yticklabels=['Background', 'Lesion'],
                annot_kws={'fontsize': 12})
    ax.set_title(f"{model_type.upper()}", fontsize=14)
    ax.set_xlabel('Predicted Label (Pixel)', fontsize=12)
    ax.set_ylabel('True Label (Pixel)', fontsize=12)

    # Add exact TP, TN, FP, FN counts as text on the heatmap cells
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j + 0.5, i + 0.5, f'{cm_labels[i][j]}\n{cm[i, j]:,}',
                    ha='center', va='center', color='black', fontsize=10)

# Hide any unused subplots
for i in range(n_models, rows * cols):
    fig.delaxes(axes[i])

plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent suptitle overlap
segmentation_cm_save_path = os.path.join(RESULTS_DIR, 'segmentation_confusion_matrices_all_models.png')
plt.savefig(segmentation_cm_save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Segmentation confusion matrices saved to: {segmentation_cm_save_path}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assuming 'all_results' and 'CLASS_NAMES' are available in the kernel
# Select a model to visualize, e.g., 'vit'
model_to_plot = 'vit'

if model_to_plot not in all_results:
    print(f"Error: Model '{model_to_plot}' not found in all_results.")
else:
    model_data = all_results[model_to_plot]

    # Extract history for plotting
    train_loss = model_data['history']['train_loss']
    val_loss = model_data['history']['val_loss']
    val_accuracy = model_data['history']['val_acc'] # train_accuracy not available in history

    # Extract confusion matrix for plotting
    cm = np.array(model_data['confusion_matrix'])

    # --- 1. Plot Epoch vs Loss (Training and Validation) ---
    plt.figure(figsize=(10, 5))
    plt.plot(train_loss, label='Training Loss', color='blue')
    plt.plot(val_loss, label='Validation Loss', color='orange')
    plt.title(f'{model_to_plot.upper()} Training and Validation Loss Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

    # --- 2. Plot Epoch vs Accuracy (Validation Only) ---
    # Note: train_accuracy is not stored in the current history structure.
    plt.figure(figsize=(10, 5))
    plt.plot(val_accuracy, label='Validation Accuracy', color='orange')
    plt.title(f'{model_to_plot.upper()} Validation Accuracy Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()

    # --- 3. Visualize Confusion Matrix ---
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{model_to_plot.upper()} Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()


### 3.9 XAI — Grad-CAM
Grad-CAM adapted for:
- **ViT**: hooks the last transformer block to get patch activations & gradients → spatial heatmap
- **U-Net (ResNet-50 encoder)**: hooks encoder layer4 → upsampled overlay

In [ ]:
# ============================================================
# XAI Method: Grad-CAM for Vision Transformer (ViT)
# ============================================================
# Grad-CAM computes gradient-weighted activation maps to highlight
# which image regions most influence the model's prediction.
# We adapt Grad-CAM for ViT by hooking the last transformer block.
# ============================================================

import cv2

class ViTGradCAM:
    """
    Grad-CAM for Vision Transformer.
    Hooks the last transformer block to get activations & gradients,
    then produces a heatmap showing which image regions drove the prediction.
    """
    def __init__(self, model):
        self.model = model
        self.activations = None
        self.gradients = None
        self._hooks = []
        self._register_hooks()

    def _register_hooks(self):
        # Hook the last transformer block of ViT
        last_block = self.model.backbone.vit.blocks[-1]

        def forward_hook(module, input, output):
            # output shape: (B, num_patches+1, embed_dim)
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self._hooks.append(last_block.register_forward_hook(forward_hook))
        self._hooks.append(last_block.register_full_backward_hook(backward_hook))

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()

    def generate(self, image_tensor, target_class=None):
        """
        Args:
            image_tensor: (1, 3, 224, 224) tensor
            target_class: int or None (uses predicted class if None)
        Returns:
            cam: (224, 224) numpy array, normalized 0-1
            pred_class: predicted class index
            pred_prob: predicted probability
        """
        self.model.eval()
        image_tensor = image_tensor.to(DEVICE).unsqueeze(0) if image_tensor.dim() == 3 else image_tensor.to(DEVICE)
        image_tensor.requires_grad_(False)

        # Forward pass
        self.model.zero_grad()
        outputs = self.model(image_tensor, return_attention=False)
        logits = outputs['cls_logits']  # Corrected key from 'classification' to 'cls_logits'
        probs = torch.softmax(logits, dim=-1)

        pred_class = logits.argmax(dim=-1).item() if target_class is None else target_class
        pred_prob = probs[0, pred_class].item()

        # Backward pass on target class score
        self.model.zero_grad()
        score = logits[0, pred_class]
        score.backward()

        # activations & gradients: (1, num_patches+1, embed_dim)
        acts = self.activations[0]        # (num_patches+1, embed_dim)
        grads = self.gradients[0]         # (num_patches+1, embed_dim)

        # Remove CLS token → patch tokens only
        acts = acts[1:]    # (196, 768)
        grads = grads[1:]  # (196, 768)

        # Global average pool gradients over embed_dim → weights per patch
        weights = grads.mean(dim=-1)  # (196,)

        # Weighted combination of activations
        cam = (weights.unsqueeze(-1) * acts).sum(dim=-1)  # (196,)
        cam = torch.relu(cam)

        # Reshape to spatial grid: 14x14
        grid = int(cam.shape[0] ** 0.5)
        cam = cam.reshape(grid, grid).cpu().numpy()

        # Normalize to 0-1
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        else:
            cam = np.zeros_like(cam)

        # Resize to 224x224
        cam = cv2.resize(cam, (224, 224), interpolation=cv2.INTER_LINEAR)
        return cam, pred_class, pred_prob


def visualize_gradcam(vit_model, test_loader, num_samples=3,
                      save_path=None, class_names=None):
    """
    Run Grad-CAM on one sample per class and plot results.
    Shows: original image | Grad-CAM heatmap | overlay | segmentation mask
    """
    if class_names is None:
        class_names = CLASS_NAMES

    gradcam = ViTGradCAM(vit_model)
    vit_model.eval()

    # Collect one sample per class
    class_samples = {}
    for batch in test_loader:
        for i in range(batch['image'].shape[0]):
            label = batch['label'][i].item()
            if label not in class_samples:
                class_samples[label] = {
                    'image': batch['image'][i],
                    'mask': batch['mask'][i],
                    'label': label
                }
        if len(class_samples) >= num_samples:
            break

    samples = [class_samples[k] for k in sorted(class_samples.keys())][:num_samples]
    n = len(samples)

    fig, axes = plt.subplots(n, 4, figsize=(18, 5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle('Grad-CAM: ViT Explanations on Breast Ultrasound',
                 fontsize=16, fontweight='bold', y=1.01)

    col_titles = ['Original Image', 'Grad-CAM Heatmap', 'Overlay (CAM on Image)', 'Ground Truth Mask']
    for col, ct in enumerate(col_titles):
        axes[0, col].set_title(ct, fontsize=13, fontweight='bold', pad=8)

    # ImageNet denormalize
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    for row, sample in enumerate(samples):
        img_t = sample['image']          # (3,224,224)
        mask  = sample['mask']           # (1,224,224)
        gt_label = sample['label']

        # Generate Grad-CAM
        with torch.enable_grad():
            cam, pred_class, pred_prob = gradcam.generate(img_t)

        # Denormalize image for display
        img_np = img_t.permute(1,2,0).cpu().numpy()
        img_np = (img_np * std + mean).clip(0, 1)
        img_gray = img_np.mean(axis=2)   # grayscale for ultrasound look

        mask_np = mask.squeeze().cpu().numpy()

        # Colormap for heatmap
        heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0

        # Overlay: blend heatmap onto grayscale image
        overlay = 0.5 * img_np + 0.5 * heatmap
        overlay = overlay.clip(0, 1)

        label_str = f"GT: {class_names[gt_label]}  |  Pred: {class_names[pred_class]} ({pred_prob:.1%})"
        color = 'green' if gt_label == pred_class else 'red'

        # Col 0: Original
        axes[row, 0].imshow(img_gray, cmap='gray')
        axes[row, 0].set_ylabel(label_str, fontsize=11, color=color, fontweight='bold')

        # Col 1: Heatmap
        axes[row, 1].imshow(img_gray, cmap='gray', alpha=0.3)
        im = axes[row, 1].imshow(cam, cmap='jet', alpha=0.7)
        plt.colorbar(im, ax=axes[row, 1], fraction=0.046, pad=0.04)

        # Col 2: Overlay
        axes[row, 2].imshow(overlay)

        # Col 3: Ground truth mask
        axes[row, 3].imshow(img_gray, cmap='gray')
        if mask_np.max() > 0:
            axes[row, 3].contour(mask_np, colors='lime', linewidths=2)
            axes[row, 3].imshow(mask_np, cmap='Greens', alpha=0.3)

        for ax in axes[row]:
            ax.axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Grad-CAM saved to: {save_path}")
    plt.show()
    gradcam.remove_hooks()
    print("\nGrad-CAM complete!")


# ── RUN GRAD-CAM ─────────────────────────────────────────
print("Loading best ViT checkpoint for Grad-CAM...")
vit_model = create_model('vit').to(DEVICE)
vit_ckpt = os.path.join(SAVE_DIR, 'vit_best.pth')
if os.path.exists(vit_ckpt):
    checkpoint = torch.load(vit_ckpt, map_location=DEVICE, weights_only=True)
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    vit_model.load_state_dict(state_dict)
    print("Checkpoint loaded successfully!")
else:
    print("No checkpoint found — using current model weights.")

gradcam_save = os.path.join(RESULTS_DIR, 'gradcam_vit.png')
visualize_gradcam(vit_model, test_loader, num_samples=3,
                  save_path=gradcam_save, class_names=CLASS_NAMES)


In [ ]:
# ============================================================
# XAI: Grad-CAM for ViT + Grad-CAM for U-Net (ResNet-50)
# ============================================================

import cv2

# ── ImageNet denormalize helper ───────────────────
def denorm(tensor):
    """(3,H,W) tensor → (H,W,3) numpy RGB image"""
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = tensor.permute(1,2,0).cpu().numpy()
    return (img * std + mean).clip(0, 1)


# ═════════════════════════════════════════════
# 1. Grad-CAM for ViT  (hooks last transformer block)
# ═════════════════════════════════════════════
class ViTGradCAM:
    """
    Hooks the last ViT transformer block.
    Computes gradient-weighted sum of patch activations → 14×14 → resize 224×224.
    """
    def __init__(self, model):
        self.model = model
        self.activations = None
        self.gradients   = None
        self._hooks      = []
        last_block = model.backbone.vit.blocks[-1]
        self._hooks.append(last_block.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o.detach())))
        self._hooks.append(last_block.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0].detach())))

    def remove_hooks(self):
        for h in self._hooks: h.remove()

    def generate(self, img_tensor, target_class=None):
        """img_tensor: (1,3,224,224).  Returns: cam(224,224), pred_class, pred_prob"""
        self.model.eval()
        x = img_tensor.to(DEVICE)
        self.model.zero_grad()
        with torch.enable_grad():
            x.requires_grad_(False)
            out     = self.model(x, return_attention=False)
            logits  = out['cls_logits']
            probs   = torch.softmax(logits, dim=-1)
            pred_cls = logits.argmax(1).item() if target_class is None else target_class
            logits[0, pred_cls].backward()

        acts  = self.activations[0, 1:]   # remove CLS → (196, 768)
        grads = self.gradients[0, 1:]     # (196, 768)
        weights = grads.mean(dim=-1)       # (196,)
        cam = torch.relu((weights.unsqueeze(-1) * acts).sum(dim=-1))  # (196,)
        grid = int(cam.shape[0] ** 0.5)
        cam  = cam.reshape(grid, grid).cpu().numpy()
        cam  = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam  = cv2.resize(cam, (224, 224), interpolation=cv2.INTER_LINEAR)
        return cam, pred_cls, probs[0, pred_cls].item()


# ═════════════════════════════════════════════
# 2. Grad-CAM for U-Net CNN encoder (hooks last ResNet layer4)
# ═════════════════════════════════════════════
class UNetGradCAM:
    """
    Standard CNN Grad-CAM on the ResNet-50 encoder's last layer (layer4).
    Computes gradient-weighted average of feature maps → 7×7 → resize 224×224.
    """
    def __init__(self, model):
        self.model = model
        self.activations = None
        self.gradients   = None
        self._hooks      = []
        target_layer = model.unet.encoder.layer4[-1] # Corrected from model.unet.encoder.model.layer4[-1]
        self._hooks.append(target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o.detach())))
        self._hooks.append(target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0].detach())))

    def remove_hooks(self):
        for h in self._hooks: h.remove()

    def generate(self, img_tensor, target_class=None):
        """img_tensor: (1,3,224,224).  Returns: cam(224,224), pred_class, pred_prob"""
        self.model.eval()
        x = img_tensor.to(DEVICE)
        self.model.zero_grad()
        with torch.enable_grad():
            out     = self.model(x, return_attention=False)
            logits  = out['cls_logits']
            probs   = torch.softmax(logits, dim=-1)
            pred_cls = logits.argmax(1).item() if target_class is None else target_class
            logits[0, pred_cls].backward()

        acts  = self.activations[0]          # (2048, 7, 7)
        grads = self.gradients[0]            # (2048, 7, 7)
        weights = grads.mean(dim=(1, 2))     # (2048,)
        cam = torch.relu((weights[:, None, None] * acts).sum(dim=0))  # (7, 7)
        cam = cam.cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam = cv2.resize(cam, (224, 224), interpolation=cv2.INTER_LINEAR)
        return cam, pred_cls, probs[0, pred_cls].item()


# ═════════════════════════════════════════════
# 3. Visualization ─ side-by-side ViT and U-Net explanations
# ═════════════════════════════════════════════
def visualize_xai_both_models(vit_model, unet_model, test_loader,
                               num_samples=3, save_path=None):
    vit_cam_gen  = ViTGradCAM(vit_model)
    unet_cam_gen = UNetGradCAM(unet_model)

    # Collect one sample per class
    class_samples = {}
    for batch in test_loader:
        for i in range(batch['image'].shape[0]):
            lbl = batch['label'][i].item()
            if lbl not in class_samples:
                class_samples[lbl] = {
                    'image': batch['image'][i],
                    'mask':  batch['mask'][i],
                    'label': lbl
                }
        if len(class_samples) >= num_samples:
            break

    samples = [class_samples[k] for k in sorted(class_samples.keys())][:num_samples]
    n = len(samples)

    fig, axes = plt.subplots(n, 5, figsize=(22, 5*n))
    if n == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle('Explainable AI ─ Grad-CAM: ViT vs U-Net',
                 fontsize=15, fontweight='bold', y=1.01)

    col_titles = ['Original', 'ViT Grad-CAM', 'ViT Overlay',
                  'U-Net Grad-CAM', 'Ground Truth Mask']
    for c, ct in enumerate(col_titles):
        axes[0, c].set_title(ct, fontsize=12, fontweight='bold')

    for row, s in enumerate(samples):
        img_t = s['image'].unsqueeze(0)      # (1,3,224,224)
        mask  = s['mask'].squeeze().numpy()  # (224,224)
        gt    = s['label']

        img_np   = denorm(s['image'])
        img_gray = img_np.mean(axis=2)

        vit_cam,  vit_pred,  vit_prob  = vit_cam_gen.generate(img_t)
        unet_cam, unet_pred, unet_prob = unet_cam_gen.generate(img_t)

        def make_heatmap(cam):
            h = cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_JET)
            return cv2.cvtColor(h, cv2.COLOR_BGR2RGB) / 255.0

        vit_heat    = make_heatmap(vit_cam)
        unet_heat   = make_heatmap(unet_cam)
        vit_overlay = (0.55*img_np + 0.45*vit_heat).clip(0,1)

        vit_color  = 'green' if vit_pred  == gt else 'red'
        unet_color = 'green' if unet_pred == gt else 'red'

        # Col 0: Original
        axes[row,0].imshow(img_gray, cmap='gray')
        axes[row,0].set_ylabel(f'GT: {CLASS_NAMES[gt]}', fontsize=11, fontweight='bold')

        # Col 1: ViT Grad-CAM
        axes[row,1].imshow(img_gray, cmap='gray', alpha=0.3)
        im1 = axes[row,1].imshow(vit_cam, cmap='jet', alpha=0.7)
        axes[row,1].set_xlabel(f'Pred: {CLASS_NAMES[vit_pred]} ({vit_prob:.0%})',
                               color=vit_color, fontsize=10, fontweight='bold')
        plt.colorbar(im1, ax=axes[row,1], fraction=0.046)

        # Col 2: ViT Overlay
        axes[row,2].imshow(vit_overlay)

        # Col 3: U-Net Grad-CAM
        axes[row,3].imshow(img_gray, cmap='gray', alpha=0.3)
        im3 = axes[row,3].imshow(unet_cam, cmap='jet', alpha=0.7)
        axes[row,3].set_xlabel(f'Pred: {CLASS_NAMES[unet_pred]} ({unet_prob:.0%})',
                               color=unet_color, fontsize=10, fontweight='bold')
        plt.colorbar(im3, ax=axes[row,3], fraction=0.046)

        # Col 4: Ground truth mask
        axes[row,4].imshow(img_gray, cmap='gray')
        if mask.max() > 0:
            axes[row,4].imshow(mask, cmap='Greens', alpha=0.4)
            axes[row,4].contour(mask, colors='lime', linewidths=2)

        for ax in axes[row]:
            ax.axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'XAI saved to: {save_path}')
    plt.show()
    vit_cam_gen.remove_hooks()
    unet_cam_gen.remove_hooks()
    print('Grad-CAM complete for both models!')


# ── LOAD BOTH MODELS AND RUN ─────────────────────────────────
print('Loading checkpoints for XAI...')

vit_model  = create_model('vit').to(DEVICE)
unet_model = create_model('unet').to(DEVICE)

for m_type, m_obj in [('vit', vit_model), ('unet', unet_model)]:
    ckpt = os.path.join(SAVE_DIR, f'{m_type}_best.pt')
    if os.path.exists(ckpt):
        m_obj.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
        print(f'  {m_type.upper()} checkpoint loaded.')
    else:
        print(f'  {m_type.upper()}: no checkpoint found, using current weights.')

visualize_xai_both_models(
    vit_model, unet_model, test_loader,
    num_samples=3,
    save_path=os.path.join(RESULTS_DIR, 'gradcam_both_models.png')
)


### 3.10 Final Summary

In [ ]:
# ============================================================
# Final Summary
# ============================================================

print('=' * 60)
print('  TRAINING COMPLETE — ViT vs U-Net (Improved Segmentation)')
print('=' * 60)
print()
print('Improvements applied:')
print('  1. FocalTversky loss (alpha=0.7, beta=0.3) — reduces FN on small lesions')
print('  2. Elastic & Grid augmentation — better shape invariance')
print('  3. U-Net: pretrained ResNet-50 encoder (ImageNet) via SMP')
print('  4. Differential LR: encoder 0.1x, decoder 1x')
print('  5. Test-Time Augmentation (TTA): avg over orig + H-flip + V-flip')
print('  6. Seg loss weight = 1.5x classification loss')
print()

if all_results:
    print(f"  {'Model':<8} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8} {'Dice':>8} {'IoU':>8}")
    print('-' * 62)
    for m in MODELS_TO_TRAIN:
        if m not in all_results: continue
        tr = all_results[m]['test_results']
        print(f"  {m.upper():<8} {tr.get('cls_accuracy',0):>8.4f} {tr.get('cls_precision',0):>8.4f} {tr.get('cls_recall',0):>8.4f} "
              f"{tr.get('cls_f1',0):>8.4f} {tr.get('seg_dice',0):>8.4f} {tr.get('seg_iou',0):>8.4f}")
    print()
    best_dice = max(all_results, key=lambda m: all_results[m]['test_results'].get('seg_dice', 0))
    best_acc  = max(all_results, key=lambda m: all_results[m]['test_results'].get('cls_accuracy', 0))
    print(f'  Best Dice    : {best_dice.upper()}  ({all_results[best_dice]["test_results"]["seg_dice"]:.4f})')
    print(f'  Best Accuracy: {best_acc.upper()}   ({all_results[best_acc]["test_results"]["cls_accuracy"]:.4f})')
    print()
    print(f'  Results saved to: {RESULTS_DIR}')
    print(f'  Checkpoints at : {SAVE_DIR}')

In [ ]:
num_algorithms = len(MODELS_TO_TRAIN)
algorithm_names = [model.upper() for model in MODELS_TO_TRAIN]

print(f"Total number of distinct algorithms identified: {num_algorithms}")
print("List of algorithms:")
for i, name in enumerate(algorithm_names):
    print(f"  {i+1}. {name}")


In [ ]:
pretrained_models = ['MultiTaskViT', 'MultiTaskUNet']
from_scratch_models = ['MultiTaskCNN (VGG-16)', 'MultiTaskResNet (ResNet-50)']

print(f"Total number of pretrained models identified: {len(pretrained_models)}")
print("List of pretrained models:")
for i, name in enumerate(pretrained_models):
    print(f"  {i+1}. {name}")

print(f"\nTotal number of models built from scratch identified: {len(from_scratch_models)}")
print("List of models built from scratch:")
for i, name in enumerate(from_scratch_models):
    print(f"  {i+1}. {name}")